In [ ]:
# Set working directory to the dataset folder (repo_root/data)
%cd ./data


In [ ]:
import pandas as pd

def process_data(file_name):
    batch_name = file_name.split('/')[-1].replace('.dat', '') + '_data'

    data = pd.read_csv(file_name, sep=' ', header=None)
    new_data = pd.DataFrame()

    for index, row in data.iterrows():
        gas_label = row[0]
        temp_dict = {"gas_label": gas_label}
        for item in row[1:]:
            if ':' in str(item):
                key, value = str(item).split(':')
                temp_dict[int(key)] = float(value)
        new_row = pd.DataFrame([temp_dict])
        new_data = pd.concat([new_data, new_row], ignore_index=True)

    new_data.reset_index(drop=True, inplace=True)

    return new_data


In [ ]:
import pandas as pd
from tensorflow.keras.utils import to_categorical

def prepare_data(data):
    Y = data['gas_label'].values - 1
    
    X = data.drop('gas_label', axis=1).values
    
    Y_encoded = to_categorical(Y, num_classes=6)
    
    return X, Y_encoded


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, precision_score, recall_score


In [ ]:
np.random.seed(1)
tf.random.set_seed(1)


In [ ]:
def create_model(input_dim):
    model = Sequential()
    model.add(Dense(100, input_dim=input_dim, activation='relu'))
    model.add(Dense(50, activation='relu'))
    model.add(Dense(20, activation='relu'))
    model.add(Dense(6, activation='softmax'))
    return model


In [ ]:
#baseline


In [ ]:
#baseline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.losses import categorical_crossentropy

all_data = []
for j in range(1, 11):
    data = process_data(f'batch{j}.dat')
    X, Y = prepare_data(data)
    all_data.append((X, Y))

X_combined = np.vstack([data[0] for data in all_data])

scaler = StandardScaler()
X_combined_scaled = scaler.fit_transform(X_combined)

start = 0
for j in range(10):
    end = start + all_data[j][0].shape[0]
    all_data[j] = (X_combined_scaled[start:end], all_data[j][1])
    start = end

results = []
detailed_accuracies = []

X_train, Y_train = all_data[0]

for j in range(2, 11):
    Xt, Yt = all_data[j-1]

    val_accuracies = []
    val_f1_scores = []
    val_precisions = []
    val_recalls = []
    
    test_accuracies = []
    test_f1_scores = []
    test_precisions = []
    test_recalls = []
    
    for experiment in range(1, 31):
        X_val, X_test, Y_val, Y_test = train_test_split(Xt, Yt, test_size=0.5, random_state=experiment)

        model = create_model(X_train.shape[1])
        model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
        model.fit(X_train, Y_train, epochs=10, batch_size=32, verbose=0)
        
        val_loss, val_accuracy = model.evaluate(X_val, Y_val, verbose=0)
        val_accuracies.append(val_accuracy)

        Y_val_pred = model.predict(X_val)
        Y_val_pred_classes = np.argmax(Y_val_pred, axis=1)
        Y_val_classes = np.argmax(Y_val, axis=1)
        val_f1 = f1_score(Y_val_classes, Y_val_pred_classes, average='macro', zero_division=1)
        val_precision = precision_score(Y_val_classes, Y_val_pred_classes, average='macro', zero_division=1)
        val_recall = recall_score(Y_val_classes, Y_val_pred_classes, average='macro', zero_division=1)
        
        val_f1_scores.append(val_f1)
        val_precisions.append(val_precision)
        val_recalls.append(val_recall)
        
        test_loss, test_accuracy = model.evaluate(X_test, Y_test, verbose=0)
        test_accuracies.append(test_accuracy)

        Y_test_pred = model.predict(X_test)
        Y_test_pred_classes = np.argmax(Y_test_pred, axis=1)
        Y_test_classes = np.argmax(Y_test, axis=1)
        test_f1 = f1_score(Y_test_classes, Y_test_pred_classes, average='macro', zero_division=1)
        test_precision = precision_score(Y_test_classes, Y_test_pred_classes, average='macro', zero_division=1)
        test_recall = recall_score(Y_test_classes, Y_test_pred_classes, average='macro', zero_division=1)
        
        test_f1_scores.append(test_f1)
        test_precisions.append(test_precision)
        test_recalls.append(test_recall)
        
        detailed_accuracies.append({
            'Model': f'model_{j}', 
            'Experiment': experiment, 
            'Val Accuracy': val_accuracy, 
            'Val F1 Score': val_f1, 
            'Val Precision': val_precision, 
            'Val Recall': val_recall, 
            'Test Accuracy': test_accuracy, 
            'Test F1 Score': test_f1, 
            'Test Precision': test_precision, 
            'Test Recall': test_recall
        })
    
    avg_val_accuracy = np.mean(val_accuracies)
    avg_val_f1 = np.mean(val_f1_scores)
    avg_val_precision = np.mean(val_precisions)
    avg_val_recall = np.mean(val_recalls)
    
    avg_test_accuracy = np.mean(test_accuracies)
    avg_test_f1 = np.mean(test_f1_scores)
    avg_test_precision = np.mean(test_precisions)
    avg_test_recall = np.mean(test_recalls)
    
    std_dev_val = np.std(val_accuracies)
    std_dev_test = np.std(test_accuracies)
    
    results.append({
        'Model': f'model_{j}', 
        'Average Val Accuracy': avg_val_accuracy, 
        'Val Standard Deviation': std_dev_val, 
        'Average Val F1 Score': avg_val_f1, 
        'Average Val Precision': avg_val_precision, 
        'Average Val Recall': avg_val_recall,
        'Average Test Accuracy': avg_test_accuracy, 
        'Test Standard Deviation': std_dev_test, 
        'Average Test F1 Score': avg_test_f1, 
        'Average Test Precision': avg_test_precision, 
        'Average Test Recall': avg_test_recall
    })

detailed_results_df = pd.DataFrame(detailed_accuracies)
summary_results_df = pd.DataFrame(results)

detailed_save_path = "outputs/.xlsx"
detailed_results_df.to_excel(detailed_save_path, index=False)

summary_save_path = "outputs/.xlsx"
summary_results_df.to_excel(summary_save_path, index=False)

print(summary_results_df)


In [ ]:
#KD


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation
from tensorflow.keras.losses import categorical_crossentropy
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import StandardScaler
input_dim = 128

teacher_model = create_model(input_dim)
student_model = create_model(input_dim)
def distillation_loss(y_true, y_pred, temperature=100):
    soft_true = tf.nn.softmax(y_true / temperature)
    soft_pred = tf.nn.softmax(y_pred / temperature)
    return categorical_crossentropy(soft_true, soft_pred)

teacher_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

student_model.compile(optimizer='adam', loss=lambda y_true, y_pred: distillation_loss(y_true, y_pred), metrics=['accuracy'])


In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.losses import categorical_crossentropy
import tensorflow as tf

def distillation_loss(y_true, y_pred, temperature):
    soft_true = tf.nn.softmax(y_true / temperature)
    soft_pred = tf.nn.softmax(y_pred / temperature)
    return categorical_crossentropy(soft_true, soft_pred)

temperature_options = [0.3, 1, 2, 3, 5, 25, 50, 100, 200]

all_data = []
for j in range(1, 11):
    data = process_data(f'batch{j}.dat')
    X, Y = prepare_data(data)
    all_data.append((X, Y))

X_combined = np.vstack([data[0] for data in all_data])

scaler = StandardScaler()
X_combined_scaled = scaler.fit_transform(X_combined)

start = 0
for j in range(10):
    end = start + all_data[j][0].shape[0]
    all_data[j] = (X_combined_scaled[start:end], all_data[j][1])
    start = end

results = []
detailed_accuracies = []

Xs, Ys = all_data[0]

for j in range(2, 11):
    Xt, Yt = all_data[j-1]

    best_accuracy = 0
    best_temperature = None
    best_val_metrics = []
    best_test_metrics = []

    for temperature in temperature_options:
        val_accuracies = []
        val_f1_scores = []
        val_precisions = []
        val_recalls = []
        test_accuracies = []
        test_f1_scores = []
        test_precisions = []
        test_recalls = []

        for experiment in range(1, 11):
            X_val, X_test, Y_val, Y_test = train_test_split(Xt, Yt, test_size=0.5, random_state=experiment)

            teacher_model = create_model(Xs.shape[1])
            student_model = create_model(Xs.shape[1])
            teacher_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
            student_model.compile(optimizer='adam', loss=lambda y_true, y_pred: distillation_loss(y_true, y_pred, temperature), metrics=['accuracy'])

            teacher_model.fit(Xs, Ys, epochs=20, batch_size=32, verbose=0)

            soft_labels_train = teacher_model.predict(Xs)
            soft_labels_val = teacher_model.predict(X_val)

            soft_labels_combined = np.vstack((soft_labels_train, soft_labels_val))
            X_combined_with_val = np.vstack((Xs, X_val))

            student_model.fit(X_combined_with_val, soft_labels_combined, epochs=20, batch_size=32, verbose=0)

            val_loss, val_accuracy = student_model.evaluate(X_val, Y_val, verbose=0)
            val_accuracies.append(val_accuracy)

            Y_val_pred = student_model.predict(X_val)
            Y_val_pred_classes = np.argmax(Y_val_pred, axis=1)
            Y_val_classes = np.argmax(Y_val, axis=1)
            val_f1 = f1_score(Y_val_classes, Y_val_pred_classes, average='macro', zero_division=1)
            val_precision = precision_score(Y_val_classes, Y_val_pred_classes, average='macro', zero_division=1)
            val_recall = recall_score(Y_val_classes, Y_val_pred_classes, average='macro', zero_division=1)

            val_f1_scores.append(val_f1)
            val_precisions.append(val_precision)
            val_recalls.append(val_recall)

            test_loss, test_accuracy = student_model.evaluate(X_test, Y_test, verbose=0)
            test_accuracies.append(test_accuracy)

            Y_test_pred = student_model.predict(X_test)
            Y_test_pred_classes = np.argmax(Y_test_pred, axis=1)
            Y_test_classes = np.argmax(Y_test, axis=1)
            test_f1 = f1_score(Y_test_classes, Y_test_pred_classes, average='macro', zero_division=1)
            test_precision = precision_score(Y_test_classes, Y_test_pred_classes, average='macro', zero_division=1)
            test_recall = recall_score(Y_test_classes, Y_test_pred_classes, average='macro', zero_division=1)

            test_f1_scores.append(test_f1)
            test_precisions.append(test_precision)
            test_recalls.append(test_recall)

            detailed_accuracies.append({'Model': f'student_model_{j}', 'Experiment': f'Temperature={temperature}', 'Val Accuracy': val_accuracy, 'Val F1 Score': val_f1, 'Val Precision': val_precision, 'Val Recall': val_recall, 'Test Accuracy': test_accuracy, 'Test F1 Score': test_f1, 'Test Precision': test_precision, 'Test Recall': test_recall, 'Repeat': experiment})

        avg_val_accuracy = np.mean(val_accuracies)
        avg_val_f1 = np.mean(val_f1_scores)
        avg_val_precision = np.mean(val_precisions)
        avg_val_recall = np.mean(val_recalls)
        avg_test_accuracy = np.mean(test_accuracies)
        avg_test_f1 = np.mean(test_f1_scores)
        avg_test_precision = np.mean(test_precisions)
        avg_test_recall = np.mean(test_recalls)

        if avg_val_accuracy > best_accuracy:
            best_accuracy = avg_val_accuracy
            best_temperature = temperature
            best_val_metrics = [{'Val Accuracy': a, 'Val F1 Score': f, 'Val Precision': p, 'Val Recall': r} for a, f, p, r in zip(val_accuracies, val_f1_scores, val_precisions, val_recalls)]
            best_test_metrics = [{'Test Accuracy': a, 'Test F1 Score': f, 'Test Precision': p, 'Test Recall': r} for a, f, p, r in zip(test_accuracies, test_f1_scores, test_precisions, test_recalls)]

    summary_results = {
        'Model': f'student_model_{j}',
        'Best Temperature': best_temperature,
        'Val Accuracy': [m['Val Accuracy'] for m in best_val_metrics],
        'Val F1 Score': [m['Val F1 Score'] for m in best_val_metrics],
        'Val Precision': [m['Val Precision'] for m in best_val_metrics],
        'Val Recall': [m['Val Recall'] for m in best_val_metrics],
        'Test Accuracy': [m['Test Accuracy'] for m in best_test_metrics],
        'Test F1 Score': [m['Test F1 Score'] for m in best_test_metrics],
        'Test Precision': [m['Test Precision'] for m in best_test_metrics],
        'Test Recall': [m['Test Recall'] for m in best_test_metrics],
        'Avg Val Accuracy': best_accuracy,
        'Avg Val F1 Score': avg_val_f1,
        'Avg Val Precision': avg_val_precision,
        'Avg Val Recall': avg_val_recall,
        'Avg Test Accuracy': avg_test_accuracy,
        'Avg Test F1 Score': avg_test_f1,
        'Avg Test Precision': avg_test_precision,
        'Avg Test Recall': avg_test_recall
    }

    results.append(summary_results)

detailed_results_df = pd.DataFrame(detailed_accuracies)
summary_results_df = pd.DataFrame(results)

detailed_results_df.to_excel("outputs/test.xlsx", index=False)
summary_results_df.to_excel("outputs/test.xlsx", index=False)

print(summary_results_df)


In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.losses import categorical_crossentropy
import tensorflow as tf

def distillation_loss(y_true, y_pred, temperature):
    soft_true = tf.nn.softmax(y_true / temperature)
    soft_pred = tf.nn.softmax(y_pred / temperature)
    return categorical_crossentropy(soft_true, soft_pred)

all_data = []
for j in range(1, 11):
    data = process_data(f'batch{j}.dat')
    X, Y = prepare_data(data)
    all_data.append((X, Y))

X_combined = np.vstack([data[0] for data in all_data])

scaler = StandardScaler()
X_combined_scaled = scaler.fit_transform(X_combined)

start = 0
for j in range(10):
    end = start + all_data[j][0].shape[0]
    all_data[j] = (X_combined_scaled[start:end], all_data[j][1])
    start = end

for j in range(2, 11):
    summary_path = "outputs/test.xlsx"
    summary_df = pd.read_excel(summary_path)

    best_temperature = summary_df.loc[summary_df['Model'] == f'student_model_{j}', 'Best Temperature'].values[0]

    results = []
    detailed_accuracies = []
    Xt, Yt = all_data[j-1]
    Xs, Ys = all_data[0]

    for experiment in range(1, 31):
        X_val, X_test, Y_val, Y_test = train_test_split(Xt, Yt, test_size=0.5, random_state=experiment)

        teacher_model = create_model(Xs.shape[1])
        student_model = create_model(Xs.shape[1])
        teacher_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
        student_model.compile(optimizer='adam', loss=lambda y_true, y_pred: distillation_loss(y_true, y_pred, best_temperature), metrics=['accuracy'])

        teacher_model.fit(Xs, Ys, epochs=20, batch_size=32, verbose=0)

        soft_labels_train = teacher_model.predict(Xs)
        soft_labels_val = teacher_model.predict(X_val)

        soft_labels_combined = np.vstack((soft_labels_train, soft_labels_val))
        X_combined_with_val = np.vstack((Xs, X_val))

        student_model.fit(X_combined_with_val, soft_labels_combined, epochs=20, batch_size=32, verbose=0)

        val_loss, val_accuracy = student_model.evaluate(X_val, Y_val, verbose=0)
        Y_val_pred = student_model.predict(X_val)
        Y_val_pred_classes = np.argmax(Y_val_pred, axis=1)
        Y_val_classes = np.argmax(Y_val, axis=1)
        val_f1 = f1_score(Y_val_classes, Y_val_pred_classes, average='macro', zero_division=1)
        val_precision = precision_score(Y_val_classes, Y_val_pred_classes, average='macro', zero_division=1)
        val_recall = recall_score(Y_val_classes, Y_val_pred_classes, average='macro', zero_division=1)

        test_loss, test_accuracy = student_model.evaluate(X_test, Y_test, verbose=0)
        Y_test_pred = student_model.predict(X_test)
        Y_test_pred_classes = np.argmax(Y_test_pred, axis=1)
        Y_test_classes = np.argmax(Y_test, axis=1)
        test_f1 = f1_score(Y_test_classes, Y_test_pred_classes, average='macro', zero_division=1)
        test_precision = precision_score(Y_test_classes, Y_test_pred_classes, average='macro', zero_division=1)
        test_recall = recall_score(Y_test_classes, Y_test_pred_classes, average='macro', zero_division=1)

        detailed_accuracies.append({
            'Model': f'student_model_{j}', 
            'Experiment': experiment, 
            'Temperature': best_temperature, 
            'Val Accuracy': val_accuracy, 
            'Val F1 Score': val_f1, 
            'Val Precision': val_precision, 
            'Val Recall': val_recall, 
            'Test Accuracy': test_accuracy, 
            'Test F1 Score': test_f1, 
            'Test Precision': test_precision, 
            'Test Recall': test_recall
        })

    detailed_results_df = pd.DataFrame(detailed_accuracies)

    detailed_save_path = f"outputs/batch_j_30_.xlsx"
    detailed_results_df.to_excel(detailed_save_path, index=False)

    print(f"Batch {j} - 30 repetitions completed and saved to {detailed_save_path}")


In [ ]:
#DRCA


In [ ]:
import numpy as np

class DRCA():
    '''
    The DRCA Class
    '''

    def __init__(self, n_components=2, alpha=None):
        self.Sw_s = None
        self.Sw_t = None
        self.mu_s = None
        self.mu_t = None
        self.alpha = alpha
        self.D_tilde = n_components

    def fit(self, Xs, Xt):
        Ns = Xs.shape[0]
        Nt = Xt.shape[0]
        D = Xs.shape[1]
        self.mu_s = np.mean(Xs, axis=0, keepdims=True)
        self.mu_t = np.mean(Xt, axis=0, keepdims=True)
        self.Sw_s = (Xs - self.mu_s).T @ (Xs - self.mu_s)
        self.Sw_t = (Xt - self.mu_t).T @ (Xt - self.mu_t)
        if self.alpha is None:
            self.alpha = Ns / Nt
        self.nominator = self.Sw_s + self.Sw_t * self.alpha
        self.denominator = (self.mu_s - self.mu_t).T @ (self.mu_s - self.mu_t)
        eigenValues, eigenVectors = np.linalg.eig(np.linalg.pinv(self.denominator) @ self.nominator)
        idx = np.abs(eigenValues).argsort()[::-1]
        self.eigenValues = eigenValues[idx]
        self.eigenVectors = eigenVectors[:, idx]
        self.W = self.eigenVectors[:, 0:self.D_tilde]

    def transform(self, X):
        return np.real(np.matmul(X, self.W))

    def fit_transform(self, Xs, Xt):
        self.fit(Xs, Xt)
        return self.transform(Xs), self.transform(Xt)


In [ ]:
# DRCA 7.30
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

all_data = []
for j in range(1, 11):
    data = process_data(f'batch{j}.dat')
    X, Y = prepare_data(data)
    all_data.append((X, Y))

X_combined = np.vstack([data[0] for data in all_data])

scaler = StandardScaler()
X_combined_scaled = scaler.fit_transform(X_combined)

start = 0
for j in range(10):
    end = start + all_data[j][0].shape[0]
    all_data[j] = (X_combined_scaled[start:end], all_data[j][1])
    start = end

n_components_options = [50, 100, 150, 200, 300, 500]
alpha_options = [0.001, 0.01, 0.1, 1, 10, 100, 1000]

results = []
detailed_accuracies = []

Xs, Ys = all_data[0]

for j in range(2, 11):
    Xt, Yt = all_data[j-1]

    best_accuracy = 0
    best_n_components = None
    best_alpha = None
    best_val_metrics = []
    best_test_metrics = []

    for n_components in n_components_options:
        for alpha in alpha_options:
            val_accuracies = []
            val_f1_scores = []
            val_precisions = []
            val_recalls = []
            test_accuracies = []
            test_f1_scores = []
            test_precisions = []
            test_recalls = []

            for experiment in range(1, 11):
                X_val, X_test, Y_val, Y_test = train_test_split(Xt, Yt, test_size=0.5, random_state=experiment)

                drca = DRCA(n_components=n_components, alpha=alpha)
                Xs_transformed, X_val_transformed = drca.fit_transform(Xs, X_val)
                Xt_transformed = drca.transform(X_test)

                model = create_model(Xs_transformed.shape[1])
                model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
                model.fit(Xs_transformed, Ys, epochs=10, batch_size=32, verbose=0)

                val_loss, val_accuracy = model.evaluate(X_val_transformed, Y_val, verbose=0)
                val_accuracies.append(val_accuracy)

                Y_val_pred = model.predict(X_val_transformed)
                Y_val_pred_classes = np.argmax(Y_val_pred, axis=1)
                Y_val_classes = np.argmax(Y_val, axis=1)
                val_f1 = f1_score(Y_val_classes, Y_val_pred_classes, average='macro', zero_division=1)
                val_precision = precision_score(Y_val_classes, Y_val_pred_classes, average='macro', zero_division=1)
                val_recall = recall_score(Y_val_classes, Y_val_pred_classes, average='macro', zero_division=1)

                val_f1_scores.append(val_f1)
                val_precisions.append(val_precision)
                val_recalls.append(val_recall)

                test_loss, test_accuracy = model.evaluate(Xt_transformed, Y_test, verbose=0)
                test_accuracies.append(test_accuracy)

                Y_test_pred = model.predict(Xt_transformed)
                Y_test_pred_classes = np.argmax(Y_test_pred, axis=1)
                Y_test_classes = np.argmax(Y_test, axis=1)
                test_f1 = f1_score(Y_test_classes, Y_test_pred_classes, average='macro', zero_division=1)
                test_precision = precision_score(Y_test_classes, Y_test_pred_classes, average='macro', zero_division=1)
                test_recall = recall_score(Y_test_classes, Y_test_pred_classes, average='macro', zero_division=1)

                test_f1_scores.append(test_f1)
                test_precisions.append(test_precision)
                test_recalls.append(test_recall)

                detailed_accuracies.append({'Model': f'drca_model_{j}', 'Experiment': f'n_components={n_components}, alpha={alpha}', 'Val Accuracy': val_accuracy, 'Val F1 Score': val_f1, 'Val Precision': val_precision, 'Val Recall': val_recall, 'Test Accuracy': test_accuracy, 'Test F1 Score': test_f1, 'Test Precision': test_precision, 'Test Recall': test_recall, 'Repeat': experiment})

            avg_val_accuracy = np.mean(val_accuracies)
            avg_val_f1 = np.mean(val_f1_scores)
            avg_val_precision = np.mean(val_precisions)
            avg_val_recall = np.mean(val_recalls)
            avg_test_accuracy = np.mean(test_accuracies)
            avg_test_f1 = np.mean(test_f1_scores)
            avg_test_precision = np.mean(test_precisions)
            avg_test_recall = np.mean(test_recalls)

            if avg_val_accuracy > best_accuracy:
                best_accuracy = avg_val_accuracy
                best_n_components = n_components
                best_alpha = alpha
                best_val_metrics = [{'Val Accuracy': a, 'Val F1 Score': f, 'Val Precision': p, 'Val Recall': r} for a, f, p, r in zip(val_accuracies, val_f1_scores, val_precisions, val_recalls)]
                best_test_metrics = [{'Test Accuracy': a, 'Test F1 Score': f, 'Test Precision': p, 'Test Recall': r} for a, f, p, r in zip(test_accuracies, test_f1_scores, test_precisions, test_recalls)]

    summary_results = {
        'Model': f'drca_model_{j}',
        'Best n_components': best_n_components,
        'Best Alpha': best_alpha,
        'Val Accuracy': [m['Val Accuracy'] for m in best_val_metrics],
        'Val F1 Score': [m['Val F1 Score'] for m in best_val_metrics],
        'Val Precision': [m['Val Precision'] for m in best_val_metrics],
        'Val Recall': [m['Val Recall'] for m in best_val_metrics],
        'Test Accuracy': [m['Test Accuracy'] for m in best_test_metrics],
        'Test F1 Score': [m['Test F1 Score'] for m in best_test_metrics],
        'Test Precision': [m['Test Precision'] for m in best_test_metrics],
        'Test Recall': [m['Test Recall'] for m in best_test_metrics],
        'Avg Val Accuracy': best_accuracy,
        'Avg Val F1 Score': avg_val_f1,
        'Avg Val Precision': avg_val_precision,
        'Avg Val Recall': avg_val_recall,
        'Avg Test Accuracy': avg_test_accuracy,
        'Avg Test F1 Score': avg_test_f1,
        'Avg Test Precision': avg_test_precision,
        'Avg Test Recall': avg_test_recall
    }

    results.append(summary_results)

detailed_results_df = pd.DataFrame(detailed_accuracies)
summary_results_df = pd.DataFrame(results)

detailed_save_path = "outputs/DRCA_.xlsx"
detailed_results_df.to_excel(detailed_save_path, index=False)

summary_save_path = "outputs/DRCA_.xlsx"
summary_results_df.to_excel(summary_save_path, index=False)

print(summary_results_df)


In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

all_data = []
for j in range(1, 11):
    data = process_data(f'batch{j}.dat')
    X, Y = prepare_data(data)
    all_data.append((X, Y))

X_combined = np.vstack([data[0] for data in all_data])

scaler = StandardScaler()
X_combined_scaled = scaler.fit_transform(X_combined)

start = 0
for j in range(10):
    end = start + all_data[j][0].shape[0]
    all_data[j] = (X_combined_scaled[start:end], all_data[j][1])
    start = end

for j in range(2, 11):
    summary_path = f"outputs/DRCA_.xlsx"
    summary_df = pd.read_excel(summary_path)

    best_n_components = summary_df.loc[summary_df['Model'] == f'drca_model_{j}', 'Best n_components'].values[0]
    best_alpha = summary_df.loc[summary_df['Model'] == f'drca_model_{j}', 'Best Alpha'].values[0]

    results = []
    detailed_accuracies = []
    Xt, Yt = all_data[j-1]
    Xs, Ys = all_data[0]

    for experiment in range(1, 31):
        X_val, X_test, Y_val, Y_test = train_test_split(Xt, Yt, test_size=0.5, random_state=experiment)

        drca = DRCA(n_components=best_n_components, alpha=best_alpha)
        Xs_transformed, X_val_transformed = drca.fit_transform(Xs, X_val)
        Xt_transformed = drca.transform(X_test)

        model = create_model(Xs_transformed.shape[1])
        model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
        model.fit(Xs_transformed, Ys, epochs=10, batch_size=32, verbose=0)

        val_loss, val_accuracy = model.evaluate(X_val_transformed, Y_val, verbose=0)
        Y_val_pred = model.predict(X_val_transformed)
        Y_val_pred_classes = np.argmax(Y_val_pred, axis=1)
        Y_val_classes = np.argmax(Y_val, axis=1)
        val_f1 = f1_score(Y_val_classes, Y_val_pred_classes, average='macro', zero_division=1)
        val_precision = precision_score(Y_val_classes, Y_val_pred_classes, average='macro', zero_division=1)
        val_recall = recall_score(Y_val_classes, Y_val_pred_classes, average='macro', zero_division=1)

        test_loss, test_accuracy = model.evaluate(Xt_transformed, Y_test, verbose=0)
        Y_test_pred = model.predict(Xt_transformed)
        Y_test_pred_classes = np.argmax(Y_test_pred, axis=1)
        Y_test_classes = np.argmax(Y_test, axis=1)
        test_f1 = f1_score(Y_test_classes, Y_test_pred_classes, average='macro', zero_division=1)
        test_precision = precision_score(Y_test_classes, Y_test_pred_classes, average='macro', zero_division=1)
        test_recall = recall_score(Y_test_classes, Y_test_pred_classes, average='macro', zero_division=1)

        detailed_accuracies.append({
            'Model': f'drca_model_{j}', 
            'Experiment': experiment, 
            'n_components': best_n_components, 
            'Alpha': best_alpha, 
            'Val Accuracy': val_accuracy, 
            'Val F1 Score': val_f1, 
            'Val Precision': val_precision, 
            'Val Recall': val_recall, 
            'Test Accuracy': test_accuracy, 
            'Test F1 Score': test_f1, 
            'Test Precision': test_precision, 
            'Test Recall': test_recall
        })

    detailed_results_df = pd.DataFrame(detailed_accuracies)

    detailed_save_path = f"outputs/DRCA_batch_j_30_.xlsx"
    detailed_results_df.to_excel(detailed_save_path, index=False)

    print(f"Batch {j} - 30 repetitions completed and saved to {detailed_save_path}")


In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.losses import categorical_crossentropy
import tensorflow as tf

def distillation_loss(y_true, y_pred, temperature):
    soft_true = tf.nn.softmax(y_true / temperature)
    soft_pred = tf.nn.softmax(y_pred / temperature)
    return categorical_crossentropy(soft_true, soft_pred)

all_data = []
for i in range(1, 11):
    data = process_data(f'batch{i}.dat')
    X, Y = prepare_data(data)
    all_data.append((X, Y))

X_combined = np.vstack([data[0] for data in all_data])

scaler = StandardScaler()
X_combined_scaled = scaler.fit_transform(X_combined)

start = 0
for i in range(10):
    end = start + all_data[i][0].shape[0]
    all_data[i] = (X_combined_scaled[start:end], all_data[i][1])
    start = end

n_components_options = [50, 100, 150, 200, 300, 500]
alpha_options = [0.001, 0.01, 0.1, 1, 10, 100, 1000]
temperature_options = [0.3, 1, 2, 3, 5, 25, 50, 100, 200]

for j in range(9, 11):
    results = []
    detailed_accuracies = []
    Xt, Yt = all_data[j-1]
    Xs = all_data[0][0]
    Ys = all_data[0][1]

    best_accuracy = 0
    best_n_components = None
    best_alpha = None
    best_temperature = None
    best_val_metrics = []
    best_test_metrics = []

    for n_components in n_components_options:
        for alpha in alpha_options:
            for temperature in temperature_options:
                val_accuracies = []
                val_f1_scores = []
                val_precisions = []
                val_recalls = []
                test_accuracies = []
                test_f1_scores = []
                test_precisions = []
                test_recalls = []

                for experiment in range(1, 4):
                    X_val, X_test, Y_val, Y_test = train_test_split(Xt, Yt, test_size=0.5, random_state=experiment)

                    drca = DRCA(n_components=n_components, alpha=alpha)
                    Xs_transformed, X_val_transformed = drca.fit_transform(Xs, X_val)
                    Xt_transformed = drca.transform(X_test)

                    input_dim = Xs_transformed.shape[1]
                    teacher_model = create_model(input_dim)
                    student_model = create_model(input_dim)
                    teacher_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
                    student_model.compile(optimizer='adam', loss=lambda y_true, y_pred: distillation_loss(y_true, y_pred, temperature), metrics=['accuracy'])
                    teacher_model.fit(Xs_transformed, Ys, epochs=15, batch_size=32, verbose=0, validation_data=(X_val_transformed, Y_val))
                    soft_labels_train = teacher_model.predict(Xs_transformed)
                    soft_labels_val = teacher_model.predict(X_val_transformed)

                    soft_labels_combined = np.vstack((soft_labels_train, soft_labels_val))
                    X_combined_with_val = np.vstack((Xs_transformed, X_val_transformed))

                    student_model.fit(X_combined_with_val, soft_labels_combined, epochs=15, batch_size=32, verbose=0)

                    val_loss, val_accuracy = student_model.evaluate(X_val_transformed, Y_val, verbose=0)
                    val_accuracies.append(val_accuracy)

                    Y_val_pred = student_model.predict(X_val_transformed)
                    Y_val_pred_classes = np.argmax(Y_val_pred, axis=1)
                    Y_val_classes = np.argmax(Y_val, axis=1)
                    val_f1 = f1_score(Y_val_classes, Y_val_pred_classes, average='macro', zero_division=1)
                    val_precision = precision_score(Y_val_classes, Y_val_pred_classes, average='macro', zero_division=1)
                    val_recall = recall_score(Y_val_classes, Y_val_pred_classes, average='macro', zero_division=1)

                    val_f1_scores.append(val_f1)
                    val_precisions.append(val_precision)
                    val_recalls.append(val_recall)

                    test_loss, test_accuracy = student_model.evaluate(Xt_transformed, Y_test, verbose=0)
                    test_accuracies.append(test_accuracy)

                    Y_test_pred = student_model.predict(Xt_transformed)
                    Y_test_pred_classes = np.argmax(Y_test_pred, axis=1)
                    Y_test_classes = np.argmax(Y_test, axis=1)
                    test_f1 = f1_score(Y_test_classes, Y_test_pred_classes, average='macro', zero_division=1)
                    test_precision = precision_score(Y_test_classes, Y_test_pred_classes, average='macro', zero_division=1)
                    test_recall = recall_score(Y_test_classes, Y_test_pred_classes, average='macro', zero_division=1)

                    test_f1_scores.append(test_f1)
                    test_precisions.append(test_precision)
                    test_recalls.append(test_recall)

                    detailed_accuracies.append({'Model': f'student_model_{j}', 'Experiment': experiment, 'n_components': n_components, 'Alpha': alpha, 'Temperature': temperature, 'Val Accuracy': val_accuracy, 'Val F1 Score': val_f1, 'Val Precision': val_precision, 'Val Recall': val_recall, 'Test Accuracy': test_accuracy, 'Test F1 Score': test_f1, 'Test Precision': test_precision, 'Test Recall': test_recall})

                avg_val_accuracy = np.mean(val_accuracies)
                avg_val_f1 = np.mean(val_f1_scores)
                avg_val_precision = np.mean(val_precisions)
                avg_val_recall = np.mean(val_recalls)

                avg_test_accuracy = np.mean(test_accuracies)
                avg_test_f1 = np.mean(test_f1_scores)
                avg_test_precision = np.mean(test_precisions)
                avg_test_recall = np.mean(test_recalls)

                if avg_val_accuracy > best_accuracy:
                    best_accuracy = avg_val_accuracy
                    best_n_components = n_components
                    best_alpha = alpha
                    best_temperature = temperature
                    best_val_metrics = [{'Val Accuracy': a, 'Val F1 Score': f, 'Val Precision': p, 'Val Recall': r} for a, f, p, r in zip(val_accuracies, val_f1_scores, val_precisions, val_recalls)]
                    best_test_metrics = [{'Test Accuracy': a, 'Test F1 Score': f, 'Test Precision': p, 'Test Recall': r} for a, f, p, r in zip(test_accuracies, test_f1_scores, test_precisions, test_recalls)]

    summary_results = {
        'Model': f'student_model_{j}',
        'Best n_components': best_n_components,
        'Best Alpha': best_alpha,
        'Best Temperature': best_temperature,
        'Val Accuracy': [m['Val Accuracy'] for m in best_val_metrics],
        'Val F1 Score': [m['Val F1 Score'] for m in best_val_metrics],
        'Val Precision': [m['Val Precision'] for m in best_val_metrics],
        'Val Recall': [m['Val Recall'] for m in best_val_metrics],
        'Test Accuracy': [m['Test Accuracy'] for m in best_test_metrics],
        'Test F1 Score': [m['Test F1 Score'] for m in best_test_metrics],
        'Test Precision': [m['Test Precision'] for m in best_test_metrics],
        'Test Recall': [m['Test Recall'] for m in best_test_metrics],
        'Avg Val Accuracy': best_accuracy,
        'Avg Val F1 Score': avg_val_f1,
        'Avg Val Precision': avg_val_precision,
        'Avg Val Recall': avg_val_recall,
        'Avg Test Accuracy': avg_test_accuracy,
        'Avg Test F1 Score': avg_test_f1,
        'Avg Test Precision': avg_test_precision,
        'Avg Test Recall': avg_test_recall
    }

    results.append(summary_results)

    detailed_results_df = pd.DataFrame(detailed_accuracies)
    summary_results_df = pd.DataFrame(results)

    detailed_save_path = f"outputs/DRCA_KD_batch_j_.xlsx"
    detailed_results_df.to_excel(detailed_save_path, index=False)

    summary_save_path = f"outputs/DRCA_KD_batch_j_.xlsx"
    summary_results_df.to_excel(summary_save_path, index=False)

    print(f"Batch {j} summary results:")
    print(summary_results_df)


In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.losses import categorical_crossentropy
import tensorflow as tf

def distillation_loss(y_true, y_pred, temperature):
    soft_true = tf.nn.softmax(y_true / temperature)
    soft_pred = tf.nn.softmax(y_pred / temperature)
    return categorical_crossentropy(soft_true, soft_pred)

all_data = []
for i in range(1, 11):
    data = process_data(f'batch{i}.dat')
    X, Y = prepare_data(data)
    all_data.append((X, Y))

X_combined = np.vstack([data[0] for data in all_data])

scaler = StandardScaler()
X_combined_scaled = scaler.fit_transform(X_combined)

start = 0
for i in range(10):
    end = start + all_data[i][0].shape[0]
    all_data[i] = (X_combined_scaled[start:end], all_data[i][1])
    start = end

for j in range(2, 11):
    summary_path = f"outputs/DRCA_KD_batch_j_.xlsx"
    summary_df = pd.read_excel(summary_path)
    
    best_n_components = summary_df['Best n_components'].values[0]
    best_alpha = summary_df['Best Alpha'].values[0]
    best_temperature = summary_df['Best Temperature'].values[0]
    
    results = []
    detailed_accuracies = []
    Xt, Yt = all_data[j-1]
    Xs = all_data[0][0]
    Ys = all_data[0][1]

    for experiment in range(1, 31):
        X_val, X_test, Y_val, Y_test = train_test_split(Xt, Yt, test_size=0.5, random_state=experiment)

        drca = DRCA(n_components=best_n_components, alpha=best_alpha)
        Xs_transformed, X_val_transformed = drca.fit_transform(Xs, X_val)
        Xt_transformed = drca.transform(X_test)

        input_dim = Xs_transformed.shape[1]
        teacher_model = create_model(input_dim)
        student_model = create_model(input_dim)
        teacher_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
        student_model.compile(optimizer='adam', loss=lambda y_true, y_pred: distillation_loss(y_true, y_pred, best_temperature), metrics=['accuracy'])
        teacher_model.fit(Xs_transformed, Ys, epochs=15, batch_size=32, verbose=0, validation_data=(X_val_transformed, Y_val))
        soft_labels_train = teacher_model.predict(Xs_transformed)
        soft_labels_val = teacher_model.predict(X_val_transformed)

        soft_labels_combined = np.vstack((soft_labels_train, soft_labels_val))
        X_combined_with_val = np.vstack((Xs_transformed, X_val_transformed))

        student_model.fit(X_combined_with_val, soft_labels_combined, epochs=15, batch_size=32, verbose=0)

        val_loss, val_accuracy = student_model.evaluate(X_val_transformed, Y_val, verbose=0)
        Y_val_pred = student_model.predict(X_val_transformed)
        Y_val_pred_classes = np.argmax(Y_val_pred, axis=1)
        Y_val_classes = np.argmax(Y_val, axis=1)
        val_f1 = f1_score(Y_val_classes, Y_val_pred_classes, average='macro', zero_division=1)
        val_precision = precision_score(Y_val_classes, Y_val_pred_classes, average='macro', zero_division=1)
        val_recall = recall_score(Y_val_classes, Y_val_pred_classes, average='macro', zero_division=1)

        test_loss, test_accuracy = student_model.evaluate(Xt_transformed, Y_test, verbose=0)
        Y_test_pred = student_model.predict(Xt_transformed)
        Y_test_pred_classes = np.argmax(Y_test_pred, axis=1)
        Y_test_classes = np.argmax(Y_test, axis=1)
        test_f1 = f1_score(Y_test_classes, Y_test_pred_classes, average='macro', zero_division=1)
        test_precision = precision_score(Y_test_classes, Y_test_pred_classes, average='macro', zero_division=1)
        test_recall = recall_score(Y_test_classes, Y_test_pred_classes, average='macro', zero_division=1)

        detailed_accuracies.append({
            'Model': f'student_model_{j}', 
            'Experiment': experiment, 
            'n_components': best_n_components, 
            'Alpha': best_alpha, 
            'Temperature': best_temperature, 
            'Val Accuracy': val_accuracy, 
            'Val F1 Score': val_f1, 
            'Val Precision': val_precision, 
            'Val Recall': val_recall, 
            'Test Accuracy': test_accuracy, 
            'Test F1 Score': test_f1, 
            'Test Precision': test_precision, 
            'Test Recall': test_recall
        })

    detailed_results_df = pd.DataFrame(detailed_accuracies)

    detailed_save_path = f"outputs/DRCA_KD_batch_j_30_.xlsx"
    detailed_results_df.to_excel(detailed_save_path, index=False)

    print(f"Batch {j} - 30 repetitions completed and saved to {detailed_save_path}")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

baseline_path = "outputs/.xlsx"
kd_summary_path = "outputs/test.xlsx"
drca_summary_path = "outputs/DRCA_.xlsx"
kd_drca_base_path = "outputs/file_20_10"

def process_data(file_path, model_prefix, test_metric, val_metric):
    df = pd.read_excel(file_path)
    df['Model'] = df['Model'].astype(str).str.replace(model_prefix, '').astype(int)

    if test_metric in df.columns:
        df[test_metric] = df[test_metric].apply(lambda x: list(map(float, str(x).strip('[]').split(', '))) if isinstance(x, str) else [x])
        df_test = df.explode(test_metric)
        df_test = df_test.rename(columns={test_metric: 'Accuracy'})
        df_test['Dataset'] = 'Test'
    else:
        df_test = pd.DataFrame()

    if val_metric in df.columns:
        df[val_metric] = df[val_metric].apply(lambda x: list(map(float, str(x).strip('[]').split(', '))) if isinstance(x, str) else [x])
        df_val = df.explode(val_metric)
        df_val = df_val.rename(columns={val_metric: 'Accuracy'})
        df_val['Dataset'] = 'Validation'
    else:
        df_val = pd.DataFrame()

    return df_test, df_val

baseline_test, baseline_val = process_data(baseline_path, 'model_', 'Test Accuracy', 'Val Accuracy')
baseline_test['Method'] = 'Baseline'
baseline_val['Method'] = 'Baseline'

kd_test, kd_val = process_data(kd_summary_path, 'student_model_', 'Test Accuracy', 'Val Accuracy')
kd_test['Method'] = 'Knowledge Distillation'
kd_val['Method'] = 'Knowledge Distillation'

drca_test, drca_val = process_data(drca_summary_path, 'drca_model_', 'Test Accuracy', 'Val Accuracy')
drca_test['Method'] = 'DRCA'
drca_val['Method'] = 'DRCA'

kd_drca_test_list = []
kd_drca_val_list = []
for batch_num in range(2, 11):
    batch_file_path = os.path.join(kd_drca_base_path, f'outputs/DRCA_KD_batch_batch_num_.xlsx')
    kd_drca_batch_test, kd_drca_batch_val = process_data(batch_file_path, 'student_model_', 'Test Accuracy', 'Val Accuracy')
    kd_drca_batch_test['Method'] = 'KD-DRCA'
    kd_drca_batch_val['Method'] = 'KD-DRCA'
    kd_drca_batch_test['Batch'] = batch_num
    kd_drca_batch_val['Batch'] = batch_num
    kd_drca_test_list.append(kd_drca_batch_test)
    kd_drca_val_list.append(kd_drca_batch_val)

kd_drca_test = pd.concat(kd_drca_test_list, ignore_index=True)
kd_drca_val = pd.concat(kd_drca_val_list, ignore_index=True)

df_combined_accuracy_test = pd.concat([baseline_test, kd_test, drca_test, kd_drca_test], ignore_index=True)

df_combined_accuracy_val = pd.concat([baseline_val, kd_val, drca_val, kd_drca_val], ignore_index=True)

plt.figure(figsize=(18, 10), dpi=80)
sns.boxplot(x='Model', y='Accuracy', data=df_combined_accuracy_test, hue='Method')

plt.xlabel('Test batch', fontsize=32)
plt.ylabel('Accuracy', fontsize=32)

plt.xticks(fontsize=28)
plt.yticks(fontsize=28)

plt.legend(title='Method', title_fontsize='24', fontsize='22', loc='lower left', bbox_to_anchor=(0.1, 0))

plt.grid(True)


output_path_test = "outputs/file_20_88"
if not os.path.exists(output_path_test):
    os.makedirs(output_path_test)
plt.savefig(output_path_test + "accuracy_test.png")

plt.tight_layout()
plt.show()

df_combined_accuracy_test.to_excel(output_path_test + "accuracy_test_data.xlsx", index=False)

print("outputs/file_20_101", output_path_test + "accuracy_test_data.xlsx")
print("outputs/file_20_102", output_path_test + "accuracy_test.png")

plt.figure(figsize=(18, 10), dpi=80)
sns.boxplot(x='Model', y='Accuracy', data=df_combined_accuracy_val, hue='Method')

plt.xlabel('Validation batch', fontsize=32)
plt.ylabel('Accuracy', fontsize=32)

plt.xticks(fontsize=28)
plt.yticks(fontsize=28)

plt.legend(title='Method', title_fontsize='24', fontsize='22', loc='lower left', bbox_to_anchor=(0.1, 0))

plt.grid(True)


output_path_val = "outputs/file_20_124"
if not os.path.exists(output_path_val):
    os.makedirs(output_path_val)
plt.savefig(output_path_val + "accuracy_val.png")

plt.tight_layout()
plt.show()

df_combined_accuracy_val.to_excel(output_path_val + "accuracy_val_data.xlsx", index=False)

print("outputs/file_20_137", output_path_val + "accuracy_val_data.xlsx")
print("outputs/file_20_138", output_path_val + "accuracy_val.png")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

baseline_path = "outputs/.xlsx"
kd_base_path = "outputs/file_21_8"
drca_base_path = "outputs/file_21_9"
kd_drca_base_path = "outputs/file_21_10"

def process_data(file_path, model_prefix, test_metric, val_metric):
    df = pd.read_excel(file_path)
    df['Model'] = df['Model'].astype(str).str.replace(model_prefix, '').astype(int)

    if test_metric in df.columns:
        df[test_metric] = df[test_metric].apply(lambda x: list(map(float, str(x).strip('[]').split(', '))) if isinstance(x, str) else [x])
        df_test = df.explode(test_metric)
        df_test = df_test.rename(columns={test_metric: 'Accuracy'})
        df_test['Dataset'] = 'Test'
    else:
        df_test = pd.DataFrame()

    if val_metric in df.columns:
        df[val_metric] = df[val_metric].apply(lambda x: list(map(float, str(x).strip('[]').split(', '))) if isinstance(x, str) else [x])
        df_val = df.explode(val_metric)
        df_val = df_val.rename(columns={val_metric: 'Accuracy'})
        df_val['Dataset'] = 'Validation'
    else:
        df_val = pd.DataFrame()

    return df_test, df_val

baseline_test, baseline_val = process_data(baseline_path, 'model_', 'Test Accuracy', 'Val Accuracy')
baseline_test['Method'] = 'Baseline'
baseline_val['Method'] = 'Baseline'

kd_test_list = []
kd_val_list = []
drca_test_list = []
drca_val_list = []
kd_drca_test_list = []
kd_drca_val_list = []

for n in range(2, 11):
    kd_file_path = os.path.join(kd_base_path, f'outputs/batch_n_30_.xlsx')
    drca_file_path = os.path.join(drca_base_path, f'outputs/DRCA_batch_n_30_.xlsx')
    kd_drca_file_path = os.path.join(kd_drca_base_path, f'outputs/DRCA_KD_batch_n_30_.xlsx')
    
    kd_batch_test, kd_batch_val = process_data(kd_file_path, 'student_model_', 'Test Accuracy', 'Val Accuracy')
    kd_batch_test['Method'] = 'Knowledge Distillation'
    kd_batch_val['Method'] = 'Knowledge Distillation'
    kd_batch_test['Batch'] = n
    kd_batch_val['Batch'] = n
    kd_test_list.append(kd_batch_test)
    kd_val_list.append(kd_batch_val)
    
    drca_batch_test, drca_batch_val = process_data(drca_file_path, 'drca_model_', 'Test Accuracy', 'Val Accuracy')
    drca_batch_test['Method'] = 'DRCA'
    drca_batch_val['Method'] = 'DRCA'
    drca_batch_test['Batch'] = n
    drca_batch_val['Batch'] = n
    drca_test_list.append(drca_batch_test)
    drca_val_list.append(drca_batch_val)
    
    kd_drca_batch_test, kd_drca_batch_val = process_data(kd_drca_file_path, 'student_model_', 'Test Accuracy', 'Val Accuracy')
    kd_drca_batch_test['Method'] = 'KD-DRCA'
    kd_drca_batch_val['Method'] = 'KD-DRCA'
    kd_drca_batch_test['Batch'] = n
    kd_drca_batch_val['Batch'] = n
    kd_drca_test_list.append(kd_drca_batch_test)
    kd_drca_val_list.append(kd_drca_batch_val)

kd_test = pd.concat(kd_test_list, ignore_index=True)
kd_val = pd.concat(kd_val_list, ignore_index=True)
drca_test = pd.concat(drca_test_list, ignore_index=True)
drca_val = pd.concat(drca_val_list, ignore_index=True)
kd_drca_test = pd.concat(kd_drca_test_list, ignore_index=True)
kd_drca_val = pd.concat(kd_drca_val_list, ignore_index=True)

df_combined_accuracy_test = pd.concat([baseline_test, kd_test, drca_test, kd_drca_test], ignore_index=True)

df_combined_accuracy_val = pd.concat([baseline_val, kd_val, drca_val, kd_drca_val], ignore_index=True)

plt.figure(figsize=(18, 10), dpi=80)
sns.boxplot(x='Model', y='Accuracy', data=df_combined_accuracy_test, hue='Method')

plt.xlabel('Test batch', fontsize=32)
plt.ylabel('Accuracy', fontsize=32)

plt.xticks(fontsize=28)
plt.yticks(fontsize=28)

plt.legend(title='Method', title_fontsize='24', fontsize='22', loc='lower left', bbox_to_anchor=(0.1, 0))

plt.grid(True)

output_path_test = "outputs/file_21_107"
if not os.path.exists(output_path_test):
    os.makedirs(output_path_test)
plt.savefig(output_path_test + "outputs/accuracy_test_30_.png")

plt.tight_layout()
plt.show()

df_combined_accuracy_test.to_excel(output_path_test + "outputs/accuracy_test_data_30_.xlsx", index=False)

print("outputs/file_21_120", output_path_test + "outputs/accuracy_test_data_30_.xlsx")
print("outputs/file_21_121", output_path_test + "outputs/accuracy_test_30_.png")

plt.figure(figsize=(18, 10), dpi=80)
sns.boxplot(x='Model', y='Accuracy', data=df_combined_accuracy_val, hue='Method')

plt.xlabel('Validation batch', fontsize=32)
plt.ylabel('Accuracy', fontsize=32)

plt.xticks(fontsize=28)
plt.yticks(fontsize=28)

plt.legend(title='Method', title_fontsize='24', fontsize='22', loc='lower left', bbox_to_anchor=(0.1, 0))

plt.grid(True)

output_path_val = "outputs/file_21_142"
if not os.path.exists(output_path_val):
    os.makedirs(output_path_val)
plt.savefig(output_path_val + "outputs/accuracy_val_30_.png")

plt.tight_layout()
plt.show()

df_combined_accuracy_val.to_excel(output_path_val + "outputs/accuracy_val_data_30_.xlsx", index=False)

print("outputs/file_21_155", output_path_val + "outputs/accuracy_val_data_30_.xlsx")
print("outputs/file_21_156", output_path_val + "outputs/accuracy_val_30_.png")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

baseline_path = "outputs/.xlsx"
kd_base_path = "outputs/file_22_8"
drca_base_path = "outputs/file_22_9"
kd_drca_base_path = "outputs/file_22_10"

metrics = ['F1 Score', 'Recall', 'Precision']

def process_data(file_path, model_prefix, metric):
    df = pd.read_excel(file_path)
    df['Model'] = df['Model'].astype(str).str.replace(model_prefix, '').astype(int)

    metric_key = f'Test {metric}'
    if metric_key in df.columns:
        df[metric_key] = df[metric_key].apply(lambda x: list(map(float, str(x).strip('[]').split(', '))) if isinstance(x, str) else [x])
        df_test = df.explode(metric_key)
        df_test = df_test.rename(columns={metric_key: metric})
        df_test['Dataset'] = 'Test'
    else:
        df_test = pd.DataFrame()

    metric_key_val = f'Val {metric}'
    if metric_key_val in df.columns:
        df[metric_key_val] = df[metric_key_val].apply(lambda x: list(map(float, str(x).strip('[]').split(', '))) if isinstance(x, str) else [x])
        df_val = df.explode(metric_key_val)
        df_val = df_val.rename(columns={metric_key_val: metric})
        df_val['Dataset'] = 'Validation'
    else:
        df_val = pd.DataFrame()

    return df_test, df_val

for metric in metrics:
    baseline_test, baseline_val = process_data(baseline_path, 'model_', metric)
    baseline_test['Method'] = 'Baseline'
    baseline_val['Method'] = 'Baseline'

    kd_test_list = []
    kd_val_list = []
    drca_test_list = []
    drca_val_list = []
    kd_drca_test_list = []
    kd_drca_val_list = []

    for n in range(2, 11):
        kd_file_path = os.path.join(kd_base_path, f'outputs/batch_n_30_.xlsx')
        drca_file_path = os.path.join(drca_base_path, f'outputs/DRCA_batch_n_30_.xlsx')
        kd_drca_file_path = os.path.join(kd_drca_base_path, f'outputs/DRCA_KD_batch_n_30_.xlsx')
        
        kd_batch_test, kd_batch_val = process_data(kd_file_path, 'student_model_', metric)
        kd_batch_test['Method'] = 'Knowledge Distillation'
        kd_batch_val['Method'] = 'Knowledge Distillation'
        kd_batch_test['Batch'] = n
        kd_batch_val['Batch'] = n
        kd_test_list.append(kd_batch_test)
        kd_val_list.append(kd_batch_val)
        
        drca_batch_test, drca_batch_val = process_data(drca_file_path, 'drca_model_', metric)
        drca_batch_test['Method'] = 'DRCA'
        drca_batch_val['Method'] = 'DRCA'
        drca_batch_test['Batch'] = n
        drca_batch_val['Batch'] = n
        drca_test_list.append(drca_batch_test)
        drca_val_list.append(drca_batch_val)
        
        kd_drca_batch_test, kd_drca_batch_val = process_data(kd_drca_file_path, 'student_model_', metric)
        kd_drca_batch_test['Method'] = 'KD-DRCA'
        kd_drca_batch_val['Method'] = 'KD-DRCA'
        kd_drca_batch_test['Batch'] = n
        kd_drca_batch_val['Batch'] = n
        kd_drca_test_list.append(kd_drca_batch_test)
        kd_drca_val_list.append(kd_drca_batch_val)

    kd_test = pd.concat(kd_test_list, ignore_index=True)
    kd_val = pd.concat(kd_val_list, ignore_index=True)
    drca_test = pd.concat(drca_test_list, ignore_index=True)
    drca_val = pd.concat(drca_val_list, ignore_index=True)
    kd_drca_test = pd.concat(kd_drca_test_list, ignore_index=True)
    kd_drca_val = pd.concat(kd_drca_val_list, ignore_index=True)

    df_combined_test = pd.concat([baseline_test, kd_test, drca_test, kd_drca_test], ignore_index=True)

    df_combined_val = pd.concat([baseline_val, kd_val, drca_val, kd_drca_val], ignore_index=True)

    plt.figure(figsize=(18, 10), dpi=80)
    sns.boxplot(x='Model', y=metric, data=df_combined_test, hue='Method')

    plt.xlabel('Test batch', fontsize=32)
    plt.ylabel(metric, fontsize=32)

    plt.xticks(fontsize=28)
    plt.yticks(fontsize=28)

    plt.legend(title='Method', title_fontsize='24', fontsize='22', loc='lower left', bbox_to_anchor=(0.1, 0))

    plt.grid(True)

    output_path_test = f"outputs/file_22_112"
    if not os.path.exists(output_path_test):
        os.makedirs(output_path_test)
    plt.savefig(output_path_test + f"outputs/metric.lower_test_30_.png")

    plt.tight_layout()
    plt.show()

    df_combined_test.to_excel(output_path_test + f"outputs/metric.lower_test_data_30_.xlsx", index=False)

    print(f"outputs/output_path_test_metric.lower_test_data_30_.xlsx")
    print(f"outputs/output_path_test_metric.lower_test_30_.png")

    plt.figure(figsize=(18, 10), dpi=80)
    sns.boxplot(x='Model', y=metric, data=df_combined_val, hue='Method')

    plt.xlabel('Validation batch', fontsize=32)
    plt.ylabel(metric, fontsize=32)

    plt.xticks(fontsize=28)
    plt.yticks(fontsize=28)

    plt.legend(title='Method', title_fontsize='24', fontsize='22', loc='lower left', bbox_to_anchor=(0.1, 0))

    plt.grid(True)

    output_path_val = f"outputs/file_22_147"
    if not os.path.exists(output_path_val):
        os.makedirs(output_path_val)
    plt.savefig(output_path_val + f"outputs/metric.lower_val_30_.png")

    plt.tight_layout()
    plt.show()

    df_combined_val.to_excel(output_path_val + f"outputs/metric.lower_val_data_30_.xlsx", index=False)

    print(f"outputs/output_path_val_metric.lower_val_data_30_.xlsx")
    print(f"outputs/output_path_val_metric.lower_val_30_.png")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

baseline_path = "outputs/.xlsx"
kd_summary_path = "outputs/test.xlsx"
drca_summary_path = "outputs/DRCA_.xlsx"
kd_drca_base_path = "outputs/file_24_10"

def process_data(file_path, model_prefix, test_metric, val_metric):
    df = pd.read_excel(file_path)
    df['Model'] = df['Model'].astype(str).str.replace(model_prefix, '').astype(int)

    if test_metric in df.columns:
        df[test_metric] = df[test_metric].apply(lambda x: list(map(float, str(x).strip('[]').split(', '))) if isinstance(x, str) else [x])
        df_test = df.explode(test_metric)
        df_test = df_test.rename(columns={test_metric: 'F1 Score'})
        df_test['Dataset'] = 'Test'
    else:
        df_test = pd.DataFrame()

    if val_metric in df.columns:
        df[val_metric] = df[val_metric].apply(lambda x: list(map(float, str(x).strip('[]').split(', '))) if isinstance(x, str) else [x])
        df_val = df.explode(val_metric)
        df_val = df_val.rename(columns={val_metric: 'F1 Score'})
        df_val['Dataset'] = 'Validation'
    else:
        df_val = pd.DataFrame()

    return df_test, df_val

baseline_test, baseline_val = process_data(baseline_path, 'model_', 'Test F1 Score', 'Val F1 Score')
baseline_test['Method'] = 'Baseline'
baseline_val['Method'] = 'Baseline'

kd_test, kd_val = process_data(kd_summary_path, 'student_model_', 'Test F1 Score', 'Val F1 Score')
kd_test['Method'] = 'Knowledge Distillation'
kd_val['Method'] = 'Knowledge Distillation'

drca_test, drca_val = process_data(drca_summary_path, 'drca_model_', 'Test F1 Score', 'Val F1 Score')
drca_test['Method'] = 'DRCA'
drca_val['Method'] = 'DRCA'

kd_drca_test_list = []
kd_drca_val_list = []
for batch_num in range(2, 11):
    batch_file_path = os.path.join(kd_drca_base_path, f'outputs/DRCA_KD_batch_batch_num_.xlsx')
    kd_drca_batch_test, kd_drca_batch_val = process_data(batch_file_path, 'student_model_', 'Test F1 Score', 'Val F1 Score')
    kd_drca_batch_test['Method'] = 'KD-DRCA'
    kd_drca_batch_val['Method'] = 'KD-DRCA'
    kd_drca_batch_test['Batch'] = batch_num
    kd_drca_batch_val['Batch'] = batch_num
    kd_drca_test_list.append(kd_drca_batch_test)
    kd_drca_val_list.append(kd_drca_batch_val)

kd_drca_test = pd.concat(kd_drca_test_list, ignore_index=True)
kd_drca_val = pd.concat(kd_drca_val_list, ignore_index=True)

df_combined_f1_test = pd.concat([baseline_test, kd_test, drca_test, kd_drca_test], ignore_index=True)

df_combined_f1_val = pd.concat([baseline_val, kd_val, drca_val, kd_drca_val], ignore_index=True)

plt.figure(figsize=(18, 10), dpi=80)
sns.boxplot(x='Model', y='F1 Score', data=df_combined_f1_test, hue='Method')

plt.xlabel('Test batch', fontsize=32)
plt.ylabel('F1 Score', fontsize=32)

plt.xticks(fontsize=28)
plt.yticks(fontsize=28)

plt.legend(title='Method', title_fontsize='24', fontsize='22', loc='lower left', bbox_to_anchor=(0.1, 0))

plt.grid(True)


output_path_test = "outputs/file_24_88"
if not os.path.exists(output_path_test):
    os.makedirs(output_path_test)
plt.savefig(output_path_test + "f1_score_test.png")

plt.tight_layout()
plt.show()

df_combined_f1_test.to_excel(output_path_test + "f1_score_test_data.xlsx", index=False)

print("outputs/file_24_101", output_path_test + "f1_score_test_data.xlsx")
print("outputs/file_24_102", output_path_test + "f1_score_test.png")

plt.figure(figsize=(18, 10), dpi=80)
sns.boxplot(x='Model', y='F1 Score', data=df_combined_f1_val, hue='Method')

plt.xlabel('Validation batch', fontsize=32)
plt.ylabel('F1 Score', fontsize=32)

plt.xticks(fontsize=28)
plt.yticks(fontsize=28)

plt.legend(title='Method', title_fontsize='24', fontsize='22', loc='lower left', bbox_to_anchor=(0.1, 0))

plt.grid(True)


output_path_val = "outputs/file_24_124"
if not os.path.exists(output_path_val):
    os.makedirs(output_path_val)
plt.savefig(output_path_val + "f1_score_val.png")

plt.tight_layout()
plt.show()

df_combined_f1_val.to_excel(output_path_val + "f1_score_val_data.xlsx", index=False)

print("outputs/file_24_137", output_path_val + "f1_score_val_data.xlsx")
print("outputs/file_24_138", output_path_val + "f1_score_val.png")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

baseline_path = "outputs/.xlsx"
kd_summary_path = "outputs/test.xlsx"
drca_summary_path = "outputs/DRCA_.xlsx"
kd_drca_base_path = "outputs/file_26_10"

def process_data(file_path, model_prefix, test_metric, val_metric):
    df = pd.read_excel(file_path)
    df['Model'] = df['Model'].astype(str).str.replace(model_prefix, '').astype(int)

    if test_metric in df.columns:
        df[test_metric] = df[test_metric].apply(lambda x: list(map(float, str(x).strip('[]').split(', '))) if isinstance(x, str) else [x])
        df_test = df.explode(test_metric)
        df_test = df_test.rename(columns={test_metric: 'Recall'})
        df_test['Dataset'] = 'Test'
    else:
        df_test = pd.DataFrame()

    if val_metric in df.columns:
        df[val_metric] = df[val_metric].apply(lambda x: list(map(float, str(x).strip('[]').split(', '))) if isinstance(x, str) else [x])
        df_val = df.explode(val_metric)
        df_val = df_val.rename(columns={val_metric: 'Recall'})
        df_val['Dataset'] = 'Validation'
    else:
        df_val = pd.DataFrame()

    return df_test, df_val

baseline_test, baseline_val = process_data(baseline_path, 'model_', 'Test Recall', 'Val Recall')
baseline_test['Method'] = 'Baseline'
baseline_val['Method'] = 'Baseline'

kd_test, kd_val = process_data(kd_summary_path, 'student_model_', 'Test Recall', 'Val Recall')
kd_test['Method'] = 'Knowledge Distillation'
kd_val['Method'] = 'Knowledge Distillation'

drca_test, drca_val = process_data(drca_summary_path, 'drca_model_', 'Test Recall', 'Val Recall')
drca_test['Method'] = 'DRCA'
drca_val['Method'] = 'DRCA'

kd_drca_test_list = []
kd_drca_val_list = []
for batch_num in range(2, 11):
    batch_file_path = os.path.join(kd_drca_base_path, f'outputs/DRCA_KD_batch_batch_num_.xlsx')
    kd_drca_batch_test, kd_drca_batch_val = process_data(batch_file_path, 'student_model_', 'Test Recall', 'Val Recall')
    kd_drca_batch_test['Method'] = 'KD-DRCA'
    kd_drca_batch_val['Method'] = 'KD-DRCA'
    kd_drca_batch_test['Batch'] = batch_num
    kd_drca_batch_val['Batch'] = batch_num
    kd_drca_test_list.append(kd_drca_batch_test)
    kd_drca_val_list.append(kd_drca_batch_val)

kd_drca_test = pd.concat(kd_drca_test_list, ignore_index=True)
kd_drca_val = pd.concat(kd_drca_val_list, ignore_index=True)

df_combined_recall_test = pd.concat([baseline_test, kd_test, drca_test, kd_drca_test], ignore_index=True)

df_combined_recall_val = pd.concat([baseline_val, kd_val, drca_val, kd_drca_val], ignore_index=True)

plt.figure(figsize=(18, 10), dpi=80)
sns.boxplot(x='Model', y='Recall', data=df_combined_recall_test, hue='Method')

plt.xlabel('Test batch', fontsize=32)
plt.ylabel('Recall', fontsize=32)

plt.xticks(fontsize=28)
plt.yticks(fontsize=28)

plt.legend(title='Method', title_fontsize='24', fontsize='22', loc='lower left', bbox_to_anchor=(0.1, 0))

plt.grid(True)


output_path_test = "outputs/file_26_88"
if not os.path.exists(output_path_test):
    os.makedirs(output_path_test)
plt.savefig(output_path_test + "recall_test.png")

plt.tight_layout()
plt.show()

df_combined_recall_test.to_excel(output_path_test + "recall_test_data.xlsx", index=False)

print("outputs/file_26_101", output_path_test + "recall_test_data.xlsx")
print("outputs/file_26_102", output_path_test + "recall_test.png")

plt.figure(figsize=(18, 10), dpi=80)
sns.boxplot(x='Model', y='Recall', data=df_combined_recall_val, hue='Method')

plt.xlabel('Validation batch', fontsize=32)
plt.ylabel('Recall', fontsize=32)

plt.xticks(fontsize=28)
plt.yticks(fontsize=28)

plt.legend(title='Method', title_fontsize='24', fontsize='22', loc='lower left', bbox_to_anchor=(0.1, 0))

plt.grid(True)


output_path_val = "outputs/file_26_124"
if not os.path.exists(output_path_val):
    os.makedirs(output_path_val)
plt.savefig(output_path_val + "recall_val.png")

plt.tight_layout()
plt.show()

df_combined_recall_val.to_excel(output_path_val + "recall_val_data.xlsx", index=False)

print("outputs/file_26_137", output_path_val + "recall_val_data.xlsx")
print("outputs/file_26_138", output_path_val + "recall_val.png")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

baseline_path = "outputs/.xlsx"
kd_summary_path = "outputs/test.xlsx"
drca_summary_path = "outputs/DRCA_.xlsx"
kd_drca_base_path = "outputs/file_28_10"

def process_data(file_path, model_prefix, test_metric, val_metric):
    df = pd.read_excel(file_path)
    df['Model'] = df['Model'].astype(str).str.replace(model_prefix, '').astype(int)

    if test_metric in df.columns:
        df[test_metric] = df[test_metric].apply(lambda x: list(map(float, str(x).strip('[]').split(', '))) if isinstance(x, str) else [x])
        df_test = df.explode(test_metric)
        df_test = df_test.rename(columns={test_metric: 'Precision'})
        df_test['Dataset'] = 'Test'
    else:
        df_test = pd.DataFrame()

    if val_metric in df.columns:
        df[val_metric] = df[val_metric].apply(lambda x: list(map(float, str(x).strip('[]').split(', '))) if isinstance(x, str) else [x])
        df_val = df.explode(val_metric)
        df_val = df_val.rename(columns={val_metric: 'Precision'})
        df_val['Dataset'] = 'Validation'
    else:
        df_val = pd.DataFrame()

    return df_test, df_val

baseline_test, baseline_val = process_data(baseline_path, 'model_', 'Test Precision', 'Val Precision')
baseline_test['Method'] = 'Baseline'
baseline_val['Method'] = 'Baseline'

kd_test, kd_val = process_data(kd_summary_path, 'student_model_', 'Test Precision', 'Val Precision')
kd_test['Method'] = 'Knowledge Distillation'
kd_val['Method'] = 'Knowledge Distillation'

drca_test, drca_val = process_data(drca_summary_path, 'drca_model_', 'Test Precision', 'Val Precision')
drca_test['Method'] = 'DRCA'
drca_val['Method'] = 'DRCA'

kd_drca_test_list = []
kd_drca_val_list = []
for batch_num in range(2, 11):
    batch_file_path = os.path.join(kd_drca_base_path, f'outputs/DRCA_KD_batch_batch_num_.xlsx')
    kd_drca_batch_test, kd_drca_batch_val = process_data(batch_file_path, 'student_model_', 'Test Precision', 'Val Precision')
    kd_drca_batch_test['Method'] = 'KD-DRCA'
    kd_drca_batch_val['Method'] = 'KD-DRCA'
    kd_drca_batch_test['Batch'] = batch_num
    kd_drca_batch_val['Batch'] = batch_num
    kd_drca_test_list.append(kd_drca_batch_test)
    kd_drca_val_list.append(kd_drca_batch_val)

kd_drca_test = pd.concat(kd_drca_test_list, ignore_index=True)
kd_drca_val = pd.concat(kd_drca_val_list, ignore_index=True)

df_combined_precision_test = pd.concat([baseline_test, kd_test, drca_test, kd_drca_test], ignore_index=True)

df_combined_precision_val = pd.concat([baseline_val, kd_val, drca_val, kd_drca_val], ignore_index=True)

plt.figure(figsize=(18, 10), dpi=80)
sns.boxplot(x='Model', y='Precision', data=df_combined_precision_test, hue='Method')

plt.xlabel('Test batch', fontsize=32)
plt.ylabel('Precision', fontsize=32)

plt.xticks(fontsize=28)
plt.yticks(fontsize=28)

plt.legend(title='Method', title_fontsize='24', fontsize='22', loc='lower left', bbox_to_anchor=(0.1, 0))

plt.grid(True)


output_path_test = "outputs/file_28_88"
if not os.path.exists(output_path_test):
    os.makedirs(output_path_test)
plt.savefig(output_path_test + "precision_test.png")

plt.tight_layout()
plt.show()

df_combined_precision_test.to_excel(output_path_test + "precision_test_data.xlsx", index=False)

print("outputs/file_28_101", output_path_test + "precision_test_data.xlsx")
print("outputs/file_28_102", output_path_test + "precision_test.png")

plt.figure(figsize=(18, 10), dpi=80)
sns.boxplot(x='Model', y='Precision', data=df_combined_precision_val, hue='Method')

plt.xlabel('Validation batch', fontsize=32)
plt.ylabel('Precision', fontsize=32)

plt.xticks(fontsize=28)
plt.yticks(fontsize=28)

plt.legend(title='Method', title_fontsize='24', fontsize='22', loc='lower left', bbox_to_anchor=(0.1, 0))

plt.grid(True)


output_path_val = "outputs/file_28_124"
if not os.path.exists(output_path_val):
    os.makedirs(output_path_val)
plt.savefig(output_path_val + "precision_val.png")

plt.tight_layout()
plt.show()

df_combined_precision_val.to_excel(output_path_val + "precision_val_data.xlsx", index=False)

print("outputs/file_28_137", output_path_val + "precision_val_data.xlsx")
print("outputs/file_28_138", output_path_val + "precision_val.png")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind
import os

baseline_paths = [
    "outputs/.xlsx",
    "outputs/.xlsx"
]

kd_paths = [
    "outputs/.xlsx",
    "outputs/test.xlsx"
]

drca_paths = [
    "outputs/DRCA_.xlsx",
    "outputs/DRCA_.xlsx"
]

kd_drca_base_paths = [
    "outputs/file_30_25",
    "outputs/file_30_26"
]

metrics = ['Test Accuracy', 'Test F1 Score', 'Test Recall', 'Test Precision', 'Val Accuracy', 'Val F1 Score', 'Val Recall', 'Val Precision']

def process_data(file_path, model_prefix, metrics):
    df = pd.read_excel(file_path)
    df['Model'] = df['Model'].astype(str).str.replace(model_prefix, '').astype(int)
    for metric in metrics:
        df[metric] = df[metric].astype(str)
        df[metric] = df[metric].apply(lambda x: list(map(float, x.strip('[]').split(', '))))
        df = df.explode(metric)
        df[metric] = df[metric].astype(float)
    return df

def compare_significance(baseline, method_data, metric):
    results = []
    for model in range(2, 11):
        baseline_scores = baseline[baseline['Model'] == model][metric].values
        method_scores = method_data[method_data['Model'] == model][metric].values
        
        if len(baseline_scores) == 0 or len(method_scores) == 0:
            results.append('No Data')
            continue
        
        baseline_scores = np.array(baseline_scores, dtype=float)
        method_scores = np.array(method_scores, dtype=float)
        
        t_stat, p_val = ttest_ind(baseline_scores, method_scores)
        mean_diff = method_scores.mean() - baseline_scores.mean()
        
        if p_val < 0.05:
            if mean_diff > 0:
                results.append('Better')
            else:
                results.append('Worse')
        else:
            results.append('No Difference')
    return results

def process_all_data():
    data = {}
    for metric in metrics:
        data[metric] = {
            "Baseline": pd.concat([process_data(path, 'model_', [metric]) for path in baseline_paths], ignore_index=True),
            "KD": pd.concat([process_data(path, 'student_model_', [metric]) for path in kd_paths], ignore_index=True),
            "DRCA": pd.concat([process_data(path, 'drca_model_', [metric]) for path in drca_paths], ignore_index=True),
            "KD_DRCA": pd.concat([process_data(os.path.join(base_path, f'outputs/DRCA_KD_batch_batch_.xlsx' if base_path == kd_drca_base_paths[0] else f'outputs/DRCA_KD_batch_batch_.xlsx'), 'student_model_', [metric])
                                  for base_path in kd_drca_base_paths for batch in range(2, 11)], ignore_index=True)
        }
    return data

def create_results_table():
    data = process_all_data()
    fig, ax = plt.subplots(figsize=(20, 12))
    
    table_data = []
    for method in ['KD', 'DRCA', 'KD_DRCA']:
        for metric in metrics:
            baseline_data = data[metric]["Baseline"]
            method_data = data[metric][method]
            results = compare_significance(baseline_data, method_data, metric)
            table_data.append([method, metric.split()[1]] + results)
    
    columns = ['Method', 'Metric'] + [f'Test Batch {i}' for i in range(2, 11)]
    colors = {'Better': 'red', 'No Difference': 'blue', 'Worse': 'green', 'No Data': 'gray'}
    cell_colors = [[colors.get(val, 'white') for val in row] for row in table_data]

    table = ax.table(cellText=table_data, cellColours=cell_colors, colLabels=columns, cellLoc='center', loc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(24)
    table.scale(2, 4)

    ax.axis('off')

    import matplotlib.patches as mpatches
    legend_patches = [mpatches.Patch(color=colors[key], label=key) for key in colors]
    plt.legend(handles=legend_patches, loc='lower right')

    plt.title('Significance Comparison of Different Methods for Various Metrics', fontsize=18)
    output_path = "outputs/file_30_113"
    plt.savefig(output_path + "significance_comparison.png")
    plt.show()

    result_df = pd.DataFrame(table_data, columns=columns)
    result_df.to_excel(output_path + "significance_comparison_data.xlsx", index=False)

create_results_table()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind


baseline_paths = [
    "outputs/.xlsx",
    "outputs/.xlsx"
]

kd_paths = [
    "outputs/.xlsx",
    "outputs/test.xlsx"
]

drca_paths = [
    "outputs/DRCA_.xlsx",
    "outputs/DRCA_.xlsx"
]

kd_drca_base_paths = [
    "outputs/file_31_24",
    "outputs/file_31_25"
]

test_metrics = ['Test Accuracy', 'Test F1 Score', 'Test Recall', 'Test Precision']
val_metrics = ['Val Accuracy', 'Val F1 Score', 'Val Recall', 'Val Precision']

def process_data(file_path, model_prefix, metrics):
    df = pd.read_excel(file_path)
    df['Model'] = df['Model'].astype(str).str.replace(model_prefix, '').astype(int)
    for metric in metrics:
        df[metric] = df[metric].astype(str)
        df[metric] = df[metric].apply(lambda x: list(map(float, x.strip('[]').split(', '))))
        df = df.explode(metric)
        df[metric] = df[metric].astype(float)
    return df

def compare_significance(baseline, method_data, metric):
    results = []
    for model in range(2, 11):
        baseline_scores = baseline[baseline['Model'] == model][metric].values
        method_scores = method_data[method_data['Model'] == model][metric].values
        
        if len(baseline_scores) == 0 or len(method_scores) == 0:
            results.append('No Data')
            continue
        
        baseline_scores = np.array(baseline_scores, dtype=float)
        method_scores = np.array(method_scores, dtype=float)
        
        t_stat, p_val = ttest_ind(baseline_scores, method_scores)
        mean_diff = method_scores.mean() - baseline_scores.mean()
        
        if p_val < 0.05:
            if mean_diff > 0:
                results.append('Better')
            else:
                results.append('Worse')
        else:
            results.append('No Difference')
    return results

def process_all_data(metrics):
    data = {}
    for metric in metrics:
        data[metric] = {
            "Baseline": pd.concat([process_data(path, 'model_', [metric]) for path in baseline_paths], ignore_index=True),
            "KD": pd.concat([process_data(path, 'student_model_', [metric]) for path in kd_paths], ignore_index=True),
            "DRCA": pd.concat([process_data(path, 'drca_model_', [metric]) for path in drca_paths], ignore_index=True),
            "KD_DRCA": pd.concat([process_data(os.path.join(base_path, f'outputs/DRCA_KD_batch_batch_.xlsx' if base_path == kd_drca_base_paths[0] else f'outputs/DRCA_KD_batch_batch_.xlsx'), 'student_model_', [metric])
                                  for base_path in kd_drca_base_paths for batch in range(2, 11)], ignore_index=True)
        }
    return data

def create_results_table(metrics, metrics_name):
    data = process_all_data(metrics)
    output_path = "outputs/file_31_84"
    
    for metric in metrics:
        fig, ax = plt.subplots(figsize=(20, 12))
        
        table_data = []
        for method in ['KD', 'DRCA', 'KD_DRCA']:
            baseline_data = data[metric]["Baseline"]
            method_data = data[metric][method]
            results = compare_significance(baseline_data, method_data, metric)
            table_data.append([method, metric.split()[1]] + results)
        
        columns = ['Method', 'Metric'] + [f'Test Batch {i}' for i in range(2, 11)]
        colors = {'Better': 'red', 'No Difference': 'blue', 'Worse': 'green', 'No Data': 'gray'}
        cell_colors = [[colors.get(val, 'white') for val in row] for row in table_data]

        table = ax.table(cellText=table_data, cellColours=cell_colors, colLabels=columns, cellLoc='center', loc='center')
        table.auto_set_font_size(False)
        table.set_fontsize(24)
        table.scale(2, 4)

        ax.axis('off')

        import matplotlib.patches as mpatches
        legend_patches = [mpatches.Patch(color=colors[key], label=key) for key in colors]
        plt.legend(handles=legend_patches, loc='lower right')

        plt.title(f'Significance Comparison of Different Methods for {metrics_name}', fontsize=18)
        plt.savefig(output_path + f"{metric.replace(' ', '_').lower()}_significance_comparison.png")
        plt.show()

        result_df = pd.DataFrame(table_data, columns=columns)
        result_df.to_excel(output_path + f"{metric.replace(' ', '_').lower()}_significance_comparison_data.xlsx", index=False)

create_results_table(test_metrics, 'Test Metrics')
create_results_table(val_metrics, 'Validation Metrics')


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind

baseline_paths = [
    "outputs/.xlsx",
    "outputs/.xlsx"
]

kd_paths = [
    "outputs/.xlsx",
    "outputs/test.xlsx"
]

drca_paths = [
    "outputs/DRCA_.xlsx",
    "outputs/DRCA_.xlsx"
]

kd_drca_base_paths = [
    "outputs/file_32_23",
    "outputs/file_32_24"
]

test_metrics = ['Test Accuracy', 'Test F1 Score', 'Test Recall', 'Test Precision']
val_metrics = ['Val Accuracy', 'Val F1 Score', 'Val Recall', 'Val Precision']

def process_data(file_path, model_prefix, metrics, offset=0):
    df = pd.read_excel(file_path)
    df['Model'] = df['Model'].astype(str).str.replace(model_prefix, '').astype(int) + offset
    df = df[df['Model'].between(2, 19)]
    for metric in metrics:
        df[metric] = df[metric].astype(str)
        df[metric] = df[metric].apply(lambda x: list(map(float, x.strip('[]').split(', '))))
        df = df.explode(metric)
        df[metric] = df[metric].astype(float)
    return df

def compare_significance(baseline, method_data, metric):
    results = {"positive": 0, "neutral": 0, "negative": 0, "total": 0}
    for model in range(2, 20):
        baseline_scores = baseline[baseline['Model'] == model][metric].values
        method_scores = method_data[method_data['Model'] == model][metric].values
        
        if len(baseline_scores) == 0 or len(method_scores) == 0:
            continue
        
        baseline_scores = np.array(baseline_scores, dtype=float)
        method_scores = np.array(method_scores, dtype=float)
        
        t_stat, p_val = ttest_ind(baseline_scores, method_scores)
        mean_diff = method_scores.mean() - baseline_scores.mean()
        
        if p_val < 0.05:
            if mean_diff > 0:
                results["positive"] += 1
            else:
                results["negative"] += 1
        else:
            results["neutral"] += 1
        
        results["total"] += 1
    return results

def process_all_data(metrics):
    data = {}
    for metric in metrics:
        data[metric] = {
            "Baseline": pd.concat([process_data(path, 'model_', [metric], offset=0 if i == 0 else 9) for i, path in enumerate(baseline_paths)], ignore_index=True),
            "KD": pd.concat([process_data(path, 'student_model_', [metric], offset=0 if i == 0 else 9) for i, path in enumerate(kd_paths)], ignore_index=True),
            "DRCA": pd.concat([process_data(path, 'drca_model_', [metric], offset=0 if i == 0 else 9) for i, path in enumerate(drca_paths)], ignore_index=True),
            "KD_DRCA": pd.concat([process_data(os.path.join(base_path, f'outputs/DRCA_KD_batch_batch_.xlsx' if base_path == kd_drca_base_paths[0] else f'outputs/DRCA_KD_batch_batch_.xlsx'), 'student_model_', [metric], offset=0 if base_path == kd_drca_base_paths[0] else 9)
                                  for base_path in kd_drca_base_paths for batch in range(2, 11)], ignore_index=True)
        }
    return data

def create_results_table(metrics, metrics_name):
    data = process_all_data(metrics)
    output_path = "outputs/file_32_85"
    
    for metric in metrics:
        table_data = []
        detailed_data = []

        for method in ['KD', 'DRCA', 'KD_DRCA']:
            baseline_data = data[metric]["Baseline"]
            method_data = data[metric][method]
            results = compare_significance(baseline_data, method_data, metric)
            table_data.append([method, results["positive"], results["neutral"], results["negative"], results["total"]])
            
            for model in range(2, 20):
                baseline_scores = baseline_data[baseline_data['Model'] == model][metric].values
                method_scores = method_data[method_data['Model'] == model][metric].values
                detailed_data.append([method, metric, model, list(baseline_scores), list(method_scores)])

        columns = ['Method', '+ (p<0.05)', '= (p>0.05)', '- (p<0.05)', 'Total']
        
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.axis('tight')
        ax.axis('off')
        table = ax.table(cellText=table_data, colLabels=columns, cellLoc='center', loc='center')
        table.auto_set_font_size(False)
        table.set_fontsize(12)
        table.scale(1.2, 1.2)
        
        plt.title(f'Significance Comparison for {metric} ({metrics_name})', fontsize=14)
        plt.savefig(output_path + f"{metric.replace(' ', '_').lower()}_significance_comparison.png")
        plt.show()

        result_df = pd.DataFrame(table_data, columns=columns)
        result_df.to_excel(output_path + f"{metric.replace(' ', '_').lower()}_significance_comparison_data.xlsx", index=False)
        
        detailed_columns = ['Method', 'Metric', 'Model', 'Baseline Scores', 'Method Scores']
        detailed_df = pd.DataFrame(detailed_data, columns=detailed_columns)
        detailed_df.to_excel(output_path + f"{metric.replace(' ', '_').lower()}_detailed_data.xlsx", index=False)

create_results_table(test_metrics, 'Test Metrics')
create_results_table(val_metrics, 'Validation Metrics')


In [ ]:
#8.25


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind

baseline_paths = [
    "outputs/.xlsx",
    "outputs/.xlsx"
]

kd_base_path_existing = "outputs/file_34_13"
drca_base_path_existing = "outputs/file_34_14"
kd_drca_base_path_existing = "outputs/file_34_15"

kd_base_path_one_to_one = "outputs/file_34_17"
drca_base_path_one_to_one = "outputs/file_34_18"
kd_drca_base_path_one_to_one = "outputs/file_34_19"

test_metrics = ['Test Accuracy', 'Test F1 Score', 'Test Recall', 'Test Precision']
val_metrics = ['Val Accuracy', 'Val F1 Score', 'Val Recall', 'Val Precision']

def process_data(file_path, model_prefix, metrics, offset=0):
    df = pd.read_excel(file_path)
    df['Model'] = df['Model'].astype(str).str.replace(model_prefix, '').astype(int) + offset
    df = df[df['Model'].between(2, 19)]
    for metric in metrics:
        df[metric] = df[metric].astype(str)
        df[metric] = df[metric].apply(lambda x: list(map(float, x.strip('[]').split(', '))))
        df = df.explode(metric)
        df[metric] = df[metric].astype(float)
    return df

def compare_significance(baseline, method_data, metric):
    results = {"positive": 0, "neutral": 0, "negative": 0, "total": 0}
    for model in range(2, 20):
        baseline_scores = baseline[baseline['Model'] == model][metric].values
        method_scores = method_data[method_data['Model'] == model][metric].values
        
        if len(baseline_scores) == 0 or len(method_scores) == 0:
            continue
        
        baseline_scores = np.array(baseline_scores, dtype=float)
        method_scores = np.array(method_scores, dtype=float)
        
        t_stat, p_val = ttest_ind(baseline_scores, method_scores)
        mean_diff = method_scores.mean() - baseline_scores.mean()
        
        if p_val < 0.05:
            if mean_diff > 0:
                results["positive"] += 1
            else:
                results["negative"] += 1
        else:
            results["neutral"] += 1
        
        results["total"] += 1
    return results

def process_all_data(metrics):
    data = {}
    for metric in metrics:
        kd_test_list, drca_test_list, kd_drca_test_list = [], [], []

        for n in range(2, 11):
            kd_file_path_one_to_one = os.path.join(kd_base_path_one_to_one, f'outputs/batch_n_30_.xlsx')
            drca_file_path_one_to_one = os.path.join(drca_base_path_one_to_one, f'outputs/DRCA_batch_n_30_.xlsx')
            kd_drca_file_path_one_to_one = os.path.join(kd_drca_base_path_one_to_one, f'outputs/DRCA_KD_batch_n_30_.xlsx')

            kd_batch_test_one_to_one = process_data(kd_file_path_one_to_one, 'student_model_', [metric], offset=0)
            drca_batch_test_one_to_one = process_data(drca_file_path_one_to_one, 'drca_model_', [metric], offset=0)
            kd_drca_batch_test_one_to_one = process_data(kd_drca_file_path_one_to_one, 'student_model_', [metric], offset=0)

            kd_test_list.append(kd_batch_test_one_to_one)
            drca_test_list.append(drca_batch_test_one_to_one)
            kd_drca_test_list.append(kd_drca_batch_test_one_to_one)

        for n in range(2, 11): 
            kd_file_path_existing = os.path.join(kd_base_path_existing, f'outputs/batch_n_30_.xlsx')
            drca_file_path_existing = os.path.join(drca_base_path_existing, f'outputs/DRCA_batch_n_30_.xlsx')
            kd_drca_file_path_existing = os.path.join(kd_drca_base_path_existing, f'outputs/DRCA_KD_batch_n_30_.xlsx')

            kd_batch_test_existing = process_data(kd_file_path_existing, 'student_model_', [metric], offset=9)
            drca_batch_test_existing = process_data(drca_file_path_existing, 'drca_model_', [metric], offset=9)
            kd_drca_batch_test_existing = process_data(kd_drca_file_path_existing, 'student_model_', [metric], offset=9)

            kd_test_list.append(kd_batch_test_existing)
            drca_test_list.append(drca_batch_test_existing)
            kd_drca_test_list.append(kd_drca_batch_test_existing)

        baseline_one_to_one = process_data(baseline_paths[1], 'model_', [metric], offset=0)
        baseline_existing = process_data(baseline_paths[0], 'model_', [metric], offset=9)

        data[metric] = {
            "Baseline": pd.concat([baseline_one_to_one, baseline_existing], ignore_index=True),
            "KD": pd.concat(kd_test_list, ignore_index=True),
            "DRCA": pd.concat(drca_test_list, ignore_index=True),
            "KD_DRCA": pd.concat(kd_drca_test_list, ignore_index=True)
        }
    return data

def create_results_table(metrics, metrics_name):
    data = process_all_data(metrics)
    output_path = "outputs/file_34_113"
    
    for metric in metrics:
        table_data = []
        detailed_data = []

        for method in ['KD', 'DRCA', 'KD_DRCA']:
            baseline_data = data[metric]["Baseline"]
            method_data = data[metric][method]
            results = compare_significance(baseline_data, method_data, metric)
            table_data.append([method, results["positive"], results["neutral"], results["negative"], results["total"]])
            
            for model in range(2, 20):
                baseline_scores = baseline_data[baseline_data['Model'] == model][metric].values
                method_scores = method_data[method_data['Model'] == model][metric].values
                detailed_data.append([method, metric, model, list(baseline_scores), list(method_scores)])

        columns = ['Method', '+ (p<0.05)', '= (p>0.05)', '- (p<0.05)', 'Total']
        
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.axis('tight')
        ax.axis('off')
        table = ax.table(cellText=table_data, colLabels=columns, cellLoc='center', loc='center')
        table.auto_set_font_size(False)
        table.set_fontsize(12)
        table.scale(1.2, 1.2)
        
        plt.title(f'Significance Comparison for {metric} ({metrics_name})', fontsize=14)
        plt.savefig(output_path + f"{metric.replace(' ', '_').lower()}_significance_comparison.png")
        plt.show()

        result_df = pd.DataFrame(table_data, columns=columns)
        result_df.to_excel(output_path + f"{metric.replace(' ', '_').lower()}_significance_comparison_data.xlsx", index=False)
        
        detailed_columns = ['Method', 'Metric', 'Model', 'Baseline Scores', 'Method Scores']
        detailed_df = pd.DataFrame(detailed_data, columns=detailed_columns)
        detailed_df.to_excel(output_path + f"{metric.replace(' ', '_').lower()}_detailed_data.xlsx", index=False)

create_results_table(test_metrics, 'Test Metrics')
create_results_table(val_metrics, 'Validation Metrics')


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind

baseline_paths = [
    "outputs/.xlsx",
    "outputs/.xlsx"
]

kd_base_path_existing = "outputs/file_36_13"
drca_base_path_existing = "outputs/file_36_14"
kd_drca_base_path_existing = "outputs/file_36_15"

kd_base_path_one_to_one = "outputs/file_36_17"
drca_base_path_one_to_one = "outputs/file_36_18"
kd_drca_base_path_one_to_one = "outputs/file_36_19"

test_metrics = ['Test Accuracy', 'Test F1 Score', 'Test Recall', 'Test Precision']
val_metrics = ['Val Accuracy', 'Val F1 Score', 'Val Recall', 'Val Precision']

def process_data(file_path, model_prefix, metrics, offset=0):
    df = pd.read_excel(file_path)
    df['Model'] = df['Model'].astype(str).str.replace(model_prefix, '').astype(int) + offset
    df = df[df['Model'].between(2, 19)]
    for metric in metrics:
        df[metric] = df[metric].astype(str)
        df[metric] = df[metric].apply(lambda x: list(map(float, x.strip('[]').split(', '))))
        df = df.explode(metric)
        df[metric] = df[metric].astype(float)
    return df

def compare_significance(baseline, method_data, metric):
    results = {"positive": 0, "neutral": 0, "negative": 0, "total": 0}
    for model in range(2, 20):
        baseline_scores = baseline[baseline['Model'] == model][metric].values
        method_scores = method_data[method_data['Model'] == model][metric].values
        
        if len(baseline_scores) == 0 or len(method_scores) == 0:
            continue
        
        baseline_scores = np.array(baseline_scores, dtype=float)
        method_scores = np.array(method_scores, dtype=float)
        
        t_stat, p_val = ttest_ind(baseline_scores, method_scores)
        mean_diff = method_scores.mean() - baseline_scores.mean()
        
        if p_val < 0.05:
            if mean_diff > 0:
                results["positive"] += 1
            else:
                results["negative"] += 1
        else:
            results["neutral"] += 1
        
        results["total"] += 1
    return results

def process_all_data(test_metrics, val_metrics):
    data = {}
    combined_metrics = [(test, val) for test, val in zip(test_metrics, val_metrics)]
    
    for test_metric, val_metric in combined_metrics:
        kd_test_list, drca_test_list, kd_drca_test_list = [], [], []

        for n in range(2, 11):
            kd_file_path_one_to_one = os.path.join(kd_base_path_one_to_one, f'outputs/batch_n_30_.xlsx')
            drca_file_path_one_to_one = os.path.join(drca_base_path_one_to_one, f'outputs/DRCA_batch_n_30_.xlsx')
            kd_drca_file_path_one_to_one = os.path.join(kd_drca_base_path_one_to_one, f'outputs/DRCA_KD_batch_n_30_.xlsx')

            kd_batch_test_one_to_one = process_data(kd_file_path_one_to_one, 'student_model_', [test_metric, val_metric], offset=0)
            drca_batch_test_one_to_one = process_data(drca_file_path_one_to_one, 'drca_model_', [test_metric, val_metric], offset=0)
            kd_drca_batch_test_one_to_one = process_data(kd_drca_file_path_one_to_one, 'student_model_', [test_metric, val_metric], offset=0)

            kd_test_list.append(kd_batch_test_one_to_one)
            drca_test_list.append(drca_batch_test_one_to_one)
            kd_drca_test_list.append(kd_drca_batch_test_one_to_one)

        for n in range(2, 11):
            kd_file_path_existing = os.path.join(kd_base_path_existing, f'outputs/batch_n_30_.xlsx')
            drca_file_path_existing = os.path.join(drca_base_path_existing, f'outputs/DRCA_batch_n_30_.xlsx')
            kd_drca_file_path_existing = os.path.join(kd_drca_base_path_existing, f'outputs/DRCA_KD_batch_n_30_.xlsx')

            kd_batch_test_existing = process_data(kd_file_path_existing, 'student_model_', [test_metric, val_metric], offset=9)
            drca_batch_test_existing = process_data(drca_file_path_existing, 'drca_model_', [test_metric, val_metric], offset=9)
            kd_drca_batch_test_existing = process_data(kd_drca_file_path_existing, 'student_model_', [test_metric, val_metric], offset=9)

            kd_test_list.append(kd_batch_test_existing)
            drca_test_list.append(drca_batch_test_existing)
            kd_drca_test_list.append(kd_drca_batch_test_existing)

        baseline_one_to_one = process_data(baseline_paths[1], 'model_', [test_metric, val_metric], offset=0)
        baseline_existing = process_data(baseline_paths[0], 'model_', [test_metric, val_metric], offset=9)

        data[test_metric] = {
            "Baseline": pd.concat([baseline_one_to_one, baseline_existing], ignore_index=True),
            "KD": pd.concat(kd_test_list, ignore_index=True),
            "DRCA": pd.concat(drca_test_list, ignore_index=True),
            "KD_DRCA": pd.concat(kd_drca_test_list, ignore_index=True)
        }
    return data
def process_task_data(data, task_num):
    if task_num == 1:
        return data[data['Model'].between(2, 10)]
    elif task_num == 2:
        return data[data['Model'].between(11, 19)]


def create_results_table(metrics, task_num, metrics_name):
    data = process_all_data(test_metrics, val_metrics)
    output_path = "outputs/file_36_124"

    for metric in metrics:
        table_data = []
        detailed_data = []

        baseline_data = process_task_data(data[metric]["Baseline"], task_num)
        kd_data = process_task_data(data[metric]["KD"], task_num)
        drca_data = process_task_data(data[metric]["DRCA"], task_num)
        kd_drca_data = process_task_data(data[metric]["KD_DRCA"], task_num)

        for method, method_data in [('KD', kd_data), ('DRCA', drca_data), ('KD_DRCA', kd_drca_data)]:
            results = compare_significance(baseline_data, method_data, metric)
            table_data.append([method, results["positive"], results["neutral"], results["negative"], results["total"]])

            for model in range(2, 20):
                baseline_scores = baseline_data[baseline_data['Model'] == model][metric].values
                method_scores = method_data[method_data['Model'] == model][metric].values
                detailed_data.append([method, metric, model, list(baseline_scores), list(method_scores)])

        columns = ['Method', '+ (p<0.05)', '= (p>0.05)', '- (p<0.05)', 'Total']
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.axis('tight')
        ax.axis('off')
        table = ax.table(cellText=table_data, colLabels=columns, cellLoc='center', loc='center')
        table.auto_set_font_size(False)
        table.set_fontsize(12)
        table.scale(1.2, 1.2)

        plt.title(f'Significance Comparison for {metric} Task {task_num} ({metrics_name})', fontsize=14)
        plt.savefig(output_path + f"{metric.replace(' ', '_').lower()}_task_{task_num}_significance_comparison.png")
        plt.show()

        result_df = pd.DataFrame(table_data, columns=columns)
        result_df.to_excel(output_path + f"{metric.replace(' ', '_').lower()}_task_{task_num}_significance_comparison_data.xlsx", index=False)

        detailed_columns = ['Method', 'Metric', 'Model', 'Baseline Scores', 'Method Scores']
        detailed_df = pd.DataFrame(detailed_data, columns=detailed_columns)
        detailed_df.to_excel(output_path + f"{metric.replace(' ', '_').lower()}_task_{task_num}_detailed_data.xlsx", index=False)


create_results_table(test_metrics, 1, 'Metrics')
create_results_table(test_metrics, 2, 'Metrics')



In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind

baseline_paths = [
    "outputs/.xlsx",
    "outputs/.xlsx"
]

kd_base_path_existing = "outputs/file_37_13"
drca_base_path_existing = "outputs/file_37_14"
kd_drca_base_path_existing = "outputs/file_37_15"

kd_base_path_one_to_one = "outputs/file_37_17"
drca_base_path_one_to_one = "outputs/file_37_18"
kd_drca_base_path_one_to_one = "outputs/file_37_19"

test_metrics = ['Test Accuracy', 'Test F1 Score', 'Test Recall', 'Test Precision']
val_metrics = ['Val Accuracy', 'Val F1 Score', 'Val Recall', 'Val Precision']

def process_data(file_path, model_prefix, test_metric, val_metric, offset=0):
    df = pd.read_excel(file_path)
    df['Model'] = df['Model'].astype(str).str.replace(model_prefix, '').astype(int) + offset
    df = df[df['Model'].between(2, 19)]

    for metric in [test_metric, val_metric]:
        df[metric] = df[metric].astype(str)
        df[metric] = df[metric].apply(lambda x: list(map(float, x.strip('[]').split(', '))))
        df = df.explode(metric)
        df[metric] = df[metric].astype(float)
    
    df['Combined Metric'] = (df[test_metric] + df[val_metric]) / 2.0
    return df[['Model', 'Combined Metric']]

def compare_significance(baseline, method_data, metric):
    results = {"positive": 0, "neutral": 0, "negative": 0, "total": 0}
    for model in range(2, 20):
        baseline_scores = baseline[baseline['Model'] == model][metric].values
        method_scores = method_data[method_data['Model'] == model][metric].values
        
        if len(baseline_scores) == 0 or len(method_scores) == 0:
            continue
        
        baseline_scores = np.array(baseline_scores, dtype=float)
        method_scores = np.array(method_scores, dtype=float)
        
        t_stat, p_val = ttest_ind(baseline_scores, method_scores)
        mean_diff = method_scores.mean() - baseline_scores.mean()
        
        if p_val < 0.05:
            if mean_diff > 0:
                results["positive"] += 1
            else:
                results["negative"] += 1
        else:
            results["neutral"] += 1
        
        results["total"] += 1
    return results

def process_all_data(test_metrics, val_metrics):
    data = {}
    combined_metrics = [(test, val) for test, val in zip(test_metrics, val_metrics)]
    
    for test_metric, val_metric in combined_metrics:
        kd_test_list, drca_test_list, kd_drca_test_list = [], [], []

        for n in range(2, 11):
            kd_file_path_one_to_one = os.path.join(kd_base_path_one_to_one, f'outputs/batch_n_30_.xlsx')
            drca_file_path_one_to_one = os.path.join(drca_base_path_one_to_one, f'outputs/DRCA_batch_n_30_.xlsx')
            kd_drca_file_path_one_to_one = os.path.join(kd_drca_base_path_one_to_one, f'outputs/DRCA_KD_batch_n_30_.xlsx')

            kd_batch_test_one_to_one = process_data(kd_file_path_one_to_one, 'student_model_', test_metric, val_metric, offset=0)
            drca_batch_test_one_to_one = process_data(drca_file_path_one_to_one, 'drca_model_', test_metric, val_metric, offset=0)
            kd_drca_batch_test_one_to_one = process_data(kd_drca_file_path_one_to_one, 'student_model_', test_metric, val_metric, offset=0)

            kd_test_list.append(kd_batch_test_one_to_one)
            drca_test_list.append(drca_batch_test_one_to_one)
            kd_drca_test_list.append(kd_drca_batch_test_one_to_one)

        for n in range(2, 11):
            kd_file_path_existing = os.path.join(kd_base_path_existing, f'outputs/batch_n_30_.xlsx')
            drca_file_path_existing = os.path.join(drca_base_path_existing, f'outputs/DRCA_batch_n_30_.xlsx')
            kd_drca_file_path_existing = os.path.join(kd_drca_base_path_existing, f'outputs/DRCA_KD_batch_n_30_.xlsx')

            kd_batch_test_existing = process_data(kd_file_path_existing, 'student_model_', test_metric, val_metric, offset=9)
            drca_batch_test_existing = process_data(drca_file_path_existing, 'drca_model_', test_metric, val_metric, offset=9)
            kd_drca_batch_test_existing = process_data(kd_drca_file_path_existing, 'student_model_', test_metric, val_metric, offset=9)

            kd_test_list.append(kd_batch_test_existing)
            drca_test_list.append(drca_batch_test_existing)
            kd_drca_test_list.append(kd_drca_batch_test_existing)

        baseline_one_to_one = process_data(baseline_paths[1], 'model_', test_metric, val_metric, offset=0)
        baseline_existing = process_data(baseline_paths[0], 'model_', test_metric, val_metric, offset=9)

        data[test_metric] = {
            "Baseline": pd.concat([baseline_one_to_one, baseline_existing], ignore_index=True),
            "KD": pd.concat(kd_test_list, ignore_index=True),
            "DRCA": pd.concat(drca_test_list, ignore_index=True),
            "KD_DRCA": pd.concat(kd_drca_test_list, ignore_index=True)
        }
    return data

def process_task_data(data, task_num):
    if task_num == 1:
        return data[data['Model'].between(2, 10)]
    elif task_num == 2:
        return data[data['Model'].between(11, 19)]

def create_results_table(metrics, task_num, metrics_name):
    data = process_all_data(test_metrics, val_metrics)
    output_path = "outputs/file_37_129"

    for metric in metrics:
        table_data = []
        detailed_data = []

        baseline_data = process_task_data(data[metric]["Baseline"], task_num)
        kd_data = process_task_data(data[metric]["KD"], task_num)
        drca_data = process_task_data(data[metric]["DRCA"], task_num)
        kd_drca_data = process_task_data(data[metric]["KD_DRCA"], task_num)

        for method, method_data in [('KD', kd_data), ('DRCA', drca_data), ('KD_DRCA', kd_drca_data)]:
            results = compare_significance(baseline_data, method_data, 'Combined Metric')
            table_data.append([method, results["positive"], results["neutral"], results["negative"], results["total"]])

            for model in range(2, 20):
                baseline_scores = baseline_data[baseline_data['Model'] == model]['Combined Metric'].values
                method_scores = method_data[method_data['Model'] == model]['Combined Metric'].values
                detailed_data.append([method, metric, model, list(baseline_scores), list(method_scores)])

        columns = ['Method', '+ (p<0.05)', '= (p>0.05)', '- (p<0.05)', 'Total']
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.axis('tight')
        ax.axis('off')
        table = ax.table(cellText=table_data, colLabels=columns, cellLoc='center', loc='center')
        table.auto_set_font_size(False)
        table.set_fontsize(12)
        table.scale(1.2, 1.2)

        plt.title(f'Significance Comparison for {metric} Task {task_num} ({metrics_name})', fontsize=14)
        plt.savefig(output_path + f"{metric.replace(' ', '_').lower()}_task_{task_num}_significance_comparison.png")
        plt.show()

        result_df = pd.DataFrame(table_data, columns=columns)
        result_df.to_excel(output_path + f"{metric.replace(' ', '_').lower()}_task_{task_num}_significance_comparison_data.xlsx", index=False)

        detailed_columns = ['Method', 'Metric', 'Model', 'Baseline Scores', 'Method Scores']
        detailed_df = pd.DataFrame(detailed_data, columns=detailed_columns)
        detailed_df.to_excel(output_path + f"{metric.replace(' ', '_').lower()}_task_{task_num}_detailed_data.xlsx", index=False)


create_results_table(test_metrics, 1, 'Metrics')
create_results_table(test_metrics, 2, 'Metrics')


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind

baseline_paths = [
    "outputs/.xlsx",
    "outputs/.xlsx"
]

kd_base_path_existing = "outputs/file_38_13"
drca_base_path_existing = "outputs/file_38_14"
kd_drca_base_path_existing = "outputs/file_38_15"

kd_base_path_one_to_one = "outputs/file_38_17"
drca_base_path_one_to_one = "outputs/file_38_18"
kd_drca_base_path_one_to_one = "outputs/file_38_19"

test_metrics = ['Test Accuracy', 'Test F1 Score', 'Test Recall', 'Test Precision']
val_metrics = ['Val Accuracy', 'Val F1 Score', 'Val Recall', 'Val Precision']

def process_data(file_path, model_prefix, metrics, offset=0):
    df = pd.read_excel(file_path)
    df['Model'] = df['Model'].astype(str).str.replace(model_prefix, '').astype(int) + offset
    df = df[df['Model'].between(2, 19)]
    for metric in metrics:
        df[metric] = df[metric].astype(str)
        df[metric] = df[metric].apply(lambda x: list(map(float, x.strip('[]').split(', '))))
        df = df.explode(metric)
        df[metric] = df[metric].astype(float)
    return df

def compare_significance(baseline, method_data, metric):
    results = {"positive": 0, "neutral": 0, "negative": 0, "total": 0}
    for model in range(2, 20):
        baseline_scores = baseline[baseline['Model'] == model][metric].values
        method_scores = method_data[method_data['Model'] == model][metric].values
        
        if len(baseline_scores) == 0 or len(method_scores) == 0:
            continue
        
        baseline_scores = np.array(baseline_scores, dtype=float)
        method_scores = np.array(method_scores, dtype=float)
        
        t_stat, p_val = ttest_ind(baseline_scores, method_scores)
        mean_diff = method_scores.mean() - baseline_scores.mean()
        
        if p_val < 0.05:
            if mean_diff > 0:
                results["positive"] += 1
            else:
                results["negative"] += 1
        else:
            results["neutral"] += 1
        
        results["total"] += 1
    return results

def process_all_data(metrics):
    data = {}
    
    for metric in metrics:
        kd_test_list, drca_test_list, kd_drca_test_list = [], [], []

        for n in range(2, 11):
            kd_file_path_one_to_one = os.path.join(kd_base_path_one_to_one, f'outputs/batch_n_30_.xlsx')
            drca_file_path_one_to_one = os.path.join(drca_base_path_one_to_one, f'outputs/DRCA_batch_n_30_.xlsx')
            kd_drca_file_path_one_to_one = os.path.join(kd_drca_base_path_one_to_one, f'outputs/DRCA_KD_batch_n_30_.xlsx')

            kd_batch_test_one_to_one = process_data(kd_file_path_one_to_one, 'student_model_', [metric], offset=0)
            drca_batch_test_one_to_one = process_data(drca_file_path_one_to_one, 'drca_model_', [metric], offset=0)
            kd_drca_batch_test_one_to_one = process_data(kd_drca_file_path_one_to_one, 'student_model_', [metric], offset=0)

            kd_test_list.append(kd_batch_test_one_to_one)
            drca_test_list.append(drca_batch_test_one_to_one)
            kd_drca_test_list.append(kd_drca_batch_test_one_to_one)

        for n in range(2, 11):
            kd_file_path_existing = os.path.join(kd_base_path_existing, f'outputs/batch_n_30_.xlsx')
            drca_file_path_existing = os.path.join(drca_base_path_existing, f'outputs/DRCA_batch_n_30_.xlsx')
            kd_drca_file_path_existing = os.path.join(kd_drca_base_path_existing, f'outputs/DRCA_KD_batch_n_30_.xlsx')

            kd_batch_test_existing = process_data(kd_file_path_existing, 'student_model_', [metric], offset=9)
            drca_batch_test_existing = process_data(drca_file_path_existing, 'drca_model_', [metric], offset=9)
            kd_drca_batch_test_existing = process_data(kd_drca_file_path_existing, 'student_model_', [metric], offset=9)

            kd_test_list.append(kd_batch_test_existing)
            drca_test_list.append(drca_batch_test_existing)
            kd_drca_test_list.append(kd_drca_batch_test_existing)

        baseline_one_to_one = process_data(baseline_paths[1], 'model_', [metric], offset=0)
        baseline_existing = process_data(baseline_paths[0], 'model_', [metric], offset=9)

        data[metric] = {
            "Baseline": pd.concat([baseline_one_to_one, baseline_existing], ignore_index=True),
            "KD": pd.concat(kd_test_list, ignore_index=True),
            "DRCA": pd.concat(drca_test_list, ignore_index=True),
            "KD_DRCA": pd.concat(kd_drca_test_list, ignore_index=True)
        }
    return data

def process_task_data(data, task_num):
    if task_num == 1:
        return data[data['Model'].between(2, 10)]
    elif task_num == 2:
        return data[data['Model'].between(11, 19)]

def create_results_table(metrics, task_num, metrics_name, dataset_type):
    data = process_all_data(metrics)
    output_path = "outputs/file_38_122"

    for metric in metrics:
        table_data = []
        detailed_data = []

        baseline_data = process_task_data(data[metric]["Baseline"], task_num)
        kd_data = process_task_data(data[metric]["KD"], task_num)
        drca_data = process_task_data(data[metric]["DRCA"], task_num)
        kd_drca_data = process_task_data(data[metric]["KD_DRCA"], task_num)

        for method, method_data in [('KD', kd_data), ('DRCA', drca_data), ('KD_DRCA', kd_drca_data)]:
            results = compare_significance(baseline_data, method_data, metric)
            table_data.append([method, results["positive"], results["neutral"], results["negative"], results["total"]])

            for model in range(2, 20):
                baseline_scores = baseline_data[baseline_data['Model'] == model][metric].values
                method_scores = method_data[method_data['Model'] == model][metric].values
                detailed_data.append([method, metric, model, list(baseline_scores), list(method_scores)])

        columns = ['Method', '+ (p<0.05)', '= (p>0.05)', '- (p<0.05)', 'Total']
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.axis('tight')
        ax.axis('off')
        table = ax.table(cellText=table_data, colLabels=columns, cellLoc='center', loc='center')
        table.auto_set_font_size(False)
        table.set_fontsize(12)
        table.scale(1.2, 1.2)
        plt.title(f'Significance Comparison for {metric} Task {task_num} ({metrics_name} - {dataset_type})', fontsize=14)
        plt.savefig(output_path + f"{metric.replace(' ', '_').lower()}_task_{task_num}_{dataset_type}_significance_comparison.png")
        plt.show()

    result_df = pd.DataFrame(table_data, columns=columns)
    result_df.to_excel(output_path + f"{metric.replace(' ', '_').lower()}_task_{task_num}_{dataset_type}_significance_comparison_data.xlsx", index=False)

    detailed_columns = ['Method', 'Metric', 'Model', 'Baseline Scores', 'Method Scores']
    detailed_df = pd.DataFrame(detailed_data, columns=detailed_columns)
    detailed_df.to_excel(output_path + f"{metric.replace(' ', '_').lower()}_task_{task_num}_{dataset_type}_detailed_data.xlsx", index=False)
create_results_table(test_metrics, 1, 'Metrics', 'Test')
create_results_table(val_metrics, 1, 'Metrics', 'Validation')
create_results_table(test_metrics, 2, 'Metrics', 'Test')
create_results_table(val_metrics, 2, 'Metrics', 'Validation')


In [ ]:
import pandas as pd

file_path = "outputs/batch_2_30_.xlsx"

df = pd.read_excel(file_path)

print(df.columns)


In [ ]:
import os
import pandas as pd
import numpy as np

baseline_paths = [
    "outputs/.xlsx",
    "outputs/.xlsx"
]

kd_base_path_existing = "outputs/file_42_11"
drca_base_path_existing = "outputs/file_42_12"
kd_drca_base_path_existing = "outputs/file_42_13"

kd_base_path_one_to_one = "outputs/file_42_15"
drca_base_path_one_to_one = "outputs/file_42_16"
kd_drca_base_path_one_to_one = "outputs/file_42_17"

def process_data(file_path, model_prefix, metrics, offset=0):
    df = pd.read_excel(file_path)
    df['Model'] = df['Model'].astype(str).str.replace(model_prefix, '').astype(int) + offset
    df = df[df['Model'].between(2, 19)]
    for metric in metrics:
        df[metric] = df[metric].astype(str)
        df[metric] = df[metric].apply(lambda x: list(map(float, x.strip('[]').split(', '))))
        df = df.explode(metric)
        df[metric] = df[metric].astype(float)
    return df

def process_all_data(metrics):
    data = {}
    for metric in metrics:
        kd_test_list, drca_test_list, kd_drca_test_list = [], [], []

        for n in range(2, 11):
            kd_file_path_one_to_one = os.path.join(kd_base_path_one_to_one, f'outputs/batch_n_30_.xlsx')
            drca_file_path_one_to_one = os.path.join(drca_base_path_one_to_one, f'outputs/DRCA_batch_n_30_.xlsx')
            kd_drca_file_path_one_to_one = os.path.join(kd_drca_base_path_one_to_one, f'outputs/DRCA_KD_batch_n_30_.xlsx')

            kd_batch_test_one_to_one = process_data(kd_file_path_one_to_one, 'student_model_', [metric], offset=0)
            drca_batch_test_one_to_one = process_data(drca_file_path_one_to_one, 'drca_model_', [metric], offset=0)
            kd_drca_batch_test_one_to_one = process_data(kd_drca_file_path_one_to_one, 'student_model_', [metric], offset=0)

            kd_test_list.append(kd_batch_test_one_to_one)
            drca_test_list.append(drca_batch_test_one_to_one)
            kd_drca_test_list.append(kd_drca_batch_test_one_to_one)

        for n in range(2, 11): 
            kd_file_path_existing = os.path.join(kd_base_path_existing, f'outputs/batch_n_30_.xlsx')
            drca_file_path_existing = os.path.join(drca_base_path_existing, f'outputs/DRCA_batch_n_30_.xlsx')
            kd_drca_file_path_existing = os.path.join(kd_drca_base_path_existing, f'outputs/DRCA_KD_batch_n_30_.xlsx')

            kd_batch_test_existing = process_data(kd_file_path_existing, 'student_model_', [metric], offset=9)
            drca_batch_test_existing = process_data(drca_file_path_existing, 'drca_model_', [metric], offset=9)
            kd_drca_batch_test_existing = process_data(kd_drca_file_path_existing, 'student_model_', [metric], offset=9)

            kd_test_list.append(kd_batch_test_existing)
            drca_test_list.append(drca_batch_test_existing)
            kd_drca_test_list.append(kd_drca_batch_test_existing)

        baseline_one_to_one = process_data(baseline_paths[1], 'model_', [metric], offset=0)
        baseline_existing = process_data(baseline_paths[0], 'model_', [metric], offset=9)

        data[metric] = {
            "Baseline": pd.concat([baseline_one_to_one, baseline_existing], ignore_index=True),
            "KD": pd.concat(kd_test_list, ignore_index=True),
            "DRCA": pd.concat(drca_test_list, ignore_index=True),
            "KD_DRCA": pd.concat(kd_drca_test_list, ignore_index=True)
        }
    return data

test_data = process_all_data(['Test Accuracy', 'Test F1 Score', 'Test Recall', 'Test Precision'])

val_data = process_all_data(['Val Accuracy', 'Val F1 Score', 'Val Recall', 'Val Precision'])


In [ ]:
pip install xlsxwriter


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

test_data = process_all_data(['Test Accuracy', 'Test F1 Score', 'Test Recall', 'Test Precision'])
val_data = process_all_data(['Val Accuracy', 'Val F1 Score', 'Val Recall', 'Val Precision'])

def save_and_plot_radar_charts(data, metrics, metrics_name, output_path):
    categories = [f"Task {i} Batch {j}" for i in range(1, 3) for j in range(2, 11)]
    
    writer = pd.ExcelWriter(os.path.join(output_path, f'{metrics_name}_median_data.xlsx'), engine='xlsxwriter')
    
    for metric in metrics:
        baseline_medians, kd_medians, drca_medians, kd_drca_medians = [], [], [], []
        
        for cat in categories:
            task_num = int(re.search(r'Task (\d+)', cat).group(1))
            batch_num = int(re.search(r'Batch (\d+)', cat).group(1))
            
            if task_num == 1:
                baseline_medians.append(data[metric]['Baseline'][(data[metric]['Baseline']['Model'] == batch_num)][metric].median())
                kd_medians.append(data[metric]['KD'][(data[metric]['KD']['Model'] == batch_num)][metric].median())
                drca_medians.append(data[metric]['DRCA'][(data[metric]['DRCA']['Model'] == batch_num)][metric].median())
                kd_drca_medians.append(data[metric]['KD_DRCA'][(data[metric]['KD_DRCA']['Model'] == batch_num)][metric].median())
            else:
                batch_num += 9
                baseline_medians.append(data[metric]['Baseline'][(data[metric]['Baseline']['Model'] == batch_num)][metric].median())
                kd_medians.append(data[metric]['KD'][(data[metric]['KD']['Model'] == batch_num)][metric].median())
                drca_medians.append(data[metric]['DRCA'][(data[metric]['DRCA']['Model'] == batch_num)][metric].median())
                kd_drca_medians.append(data[metric]['KD_DRCA'][(data[metric]['KD_DRCA']['Model'] == batch_num)][metric].median())

        baseline_medians = np.array(baseline_medians)
        kd_medians = np.array(kd_medians) / baseline_medians
        drca_medians = np.array(drca_medians) / baseline_medians
        kd_drca_medians = np.array(kd_drca_medians) / baseline_medians
        baseline_medians = np.ones_like(baseline_medians)

        df = pd.DataFrame({
            'Category': categories,
            'Baseline': baseline_medians,
            'KD': kd_medians,
            'DRCA': drca_medians,
            'KD-DRCA': kd_drca_medians
        })
        df.to_excel(writer, sheet_name=metric.replace(' ', '_'), index=False)

        N = len(categories)
        angles = [n / float(N) * 2 * np.pi for n in range(N)]
        angles += angles[:1]

        fig, ax = plt.subplots(figsize=(10, 10), dpi=300, subplot_kw=dict(polar=True))
        
        ax.plot(angles, np.concatenate([baseline_medians, baseline_medians[:1]]), linewidth=4, linestyle='solid', label='Baseline')
        ax.plot(angles, np.concatenate([kd_medians, kd_medians[:1]]), linewidth=4, linestyle='dashed', label='KD')
        ax.plot(angles, np.concatenate([drca_medians, drca_medians[:1]]), linewidth=4, linestyle='dotted', label='DRCA')
        ax.plot(angles, np.concatenate([kd_drca_medians, kd_drca_medians[:1]]), linewidth=4, linestyle='dashdot', label='KD-DRCA')

        ax.set_theta_offset(np.pi / 2)
        ax.set_theta_direction(-1)
        plt.xticks(angles[:-1], categories, fontsize=20)
        ax.set_rlabel_position(0)
        plt.yticks([0.8, 1.0, 1.3], ["0.8", "1.0", "1.3"], color="grey", size=15)
        plt.ylim(0.8, 1.3)

        plt.title(f'{metric}', size=40, color='black', y=1.1)
        plt.legend(loc='upper right', bbox_to_anchor=(1.2, 1.2), prop={'size': 20})

        plt.savefig(os.path.join(output_path, f'{metric.replace(" ", "_").lower()}_radar_chart.png'), bbox_inches='tight')
        plt.close()

    writer.close()

output_path = "outputs/file_45_86"

save_and_plot_radar_charts(test_data, ['Test Accuracy', 'Test F1 Score', 'Test Recall', 'Test Precision'], 'Test Metrics', output_path)

save_and_plot_radar_charts(val_data, ['Val Accuracy', 'Val F1 Score', 'Val Recall', 'Val Precision'], 'Validation Metrics', output_path)


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

test_data = process_all_data(['Test Accuracy', 'Test F1 Score', 'Test Recall', 'Test Precision'])
val_data = process_all_data(['Val Accuracy', 'Val F1 Score', 'Val Recall', 'Val Precision'])

def save_and_plot_radar_charts(data, metrics, metrics_name, output_path):
    categories = [f"Task {i} Batch {j}" for i in range(1, 3) for j in range(2, 11)]
    
    writer = pd.ExcelWriter(os.path.join(output_path, f'{metrics_name}_mean_data.xlsx'), engine='xlsxwriter')
    
    for metric in metrics:
        baseline_means, kd_means, drca_means, kd_drca_means = [], [], [], []
        
        for cat in categories:
            task_num = int(re.search(r'Task (\d+)', cat).group(1))
            batch_num = int(re.search(r'Batch (\d+)', cat).group(1))
            
            if task_num == 1:
                baseline_means.append(data[metric]['Baseline'][(data[metric]['Baseline']['Model'] == batch_num)][metric].mean())
                kd_means.append(data[metric]['KD'][(data[metric]['KD']['Model'] == batch_num)][metric].mean())
                drca_means.append(data[metric]['DRCA'][(data[metric]['DRCA']['Model'] == batch_num)][metric].mean())
                kd_drca_means.append(data[metric]['KD_DRCA'][(data[metric]['KD_DRCA']['Model'] == batch_num)][metric].mean())
            else:
                batch_num += 9
                baseline_means.append(data[metric]['Baseline'][(data[metric]['Baseline']['Model'] == batch_num)][metric].mean())
                kd_means.append(data[metric]['KD'][(data[metric]['KD']['Model'] == batch_num)][metric].mean())
                drca_means.append(data[metric]['DRCA'][(data[metric]['DRCA']['Model'] == batch_num)][metric].mean())
                kd_drca_means.append(data[metric]['KD_DRCA'][(data[metric]['KD_DRCA']['Model'] == batch_num)][metric].mean())

        baseline_means = np.array(baseline_means)
        kd_means = np.array(kd_means) / baseline_means
        drca_means = np.array(drca_means) / baseline_means
        kd_drca_means = np.array(kd_drca_means) / baseline_means
        baseline_means = np.ones_like(baseline_means)

        df = pd.DataFrame({
            'Category': categories,
            'Baseline': baseline_means,
            'KD': kd_means,
            'DRCA': drca_means,
            'KD-DRCA': kd_drca_means
        })
        df.to_excel(writer, sheet_name=metric.replace(' ', '_'), index=False)

        N = len(categories)
        angles = [n / float(N) * 2 * np.pi for n in range(N)]
        angles += angles[:1]

        fig, ax = plt.subplots(figsize=(10, 10), dpi=300, subplot_kw=dict(polar=True))
        
        ax.plot(angles, np.concatenate([baseline_means, baseline_means[:1]]), linewidth=4, linestyle='solid', label='Baseline')
        ax.plot(angles, np.concatenate([kd_means, kd_means[:1]]), linewidth=4, linestyle='dashed', label='KD')
        ax.plot(angles, np.concatenate([drca_means, drca_means[:1]]), linewidth=4, linestyle='dotted', label='DRCA')
        ax.plot(angles, np.concatenate([kd_drca_means, kd_drca_means[:1]]), linewidth=4, linestyle='dashdot', label='KD-DRCA')

        ax.set_theta_offset(np.pi / 2)
        ax.set_theta_direction(-1)
        plt.xticks(angles[:-1], categories, fontsize=20)
        ax.set_rlabel_position(0)
        plt.yticks([0.8, 1.0, 1.3], ["0.8", "1.0", "1.3"], color="grey", size=15)
        plt.ylim(0.8, 1.3)

        plt.title(f'{metric}', size=40, color='black', y=1.1)
        plt.legend(loc='upper right', bbox_to_anchor=(1.2, 1.2), prop={'size': 20})

        plt.savefig(os.path.join(output_path, f'{metric.replace(" ", "_").lower()}_mean_radar_chart.png'), bbox_inches='tight')
        plt.close()

    writer.close()

output_path = "outputs/file_47_86"

save_and_plot_radar_charts(test_data, ['Test Accuracy', 'Test F1 Score', 'Test Recall', 'Test Precision'], 'Test Metrics', output_path)

save_and_plot_radar_charts(val_data, ['Val Accuracy', 'Val F1 Score', 'Val Recall', 'Val Precision'], 'Validation Metrics', output_path)


In [ ]:
from PIL import Image

def combine_images():
    save_path_combined = "outputs/file_49_5"
    if not os.path.exists(save_path_combined):
        os.makedirs(save_path_combined)
    
    median_path = "outputs/file_49_10"
    avg_path = "outputs/file_49_12"
    
    median_images = sorted([f for f in os.listdir(median_path) if f.endswith("_radar_chart_median.png")])
    avg_images = sorted([f for f in os.listdir(avg_path) if f.endswith("_radar_chart_mean.png")])
    
    print("Median Images:", median_images)
    print("Average Images:", avg_images)
    
    assert len(median_images) == len(avg_images), "outputs/file_49_23"
    
    img_sample = Image.open(os.path.join(median_path, median_images[0]))
    img_width, img_height = img_sample.size
    total_width = img_width * 2
    total_height = img_height * len(median_images)
    
    combined_image = Image.new('RGB', (total_width, total_height))
    
    for i in range(len(median_images)):
        median_img = Image.open(os.path.join(median_path, median_images[i]))
        avg_img = Image.open(os.path.join(avg_path, avg_images[i]))
        
        combined_image.paste(median_img, (0, i * img_height))
        combined_image.paste(avg_img, (img_width, i * img_height))
    
    combined_image.save(os.path.join(save_path_combined, "combined_radar_charts.png"))
    combined_image.show()

combine_images()


In [ ]:
import os
import matplotlib.pyplot as plt
from PIL import Image

def combine_task1_images():
    image_paths = [
        "outputs/accuracy_test.png",
        "outputs/accuracy_val.png",
        "outputs/f1_score_test.png",
        "outputs/f1_score_val.png",
        "outputs/precision_test.png",
        "outputs/precision_val.png",
        "outputs/recall_test.png",
        "outputs/recall_val.png"
    ]
    
    images = [Image.open(image) for image in image_paths]
    widths, heights = zip(*(img.size for img in images))

    total_width = max(widths) * 2
    total_height = max(heights) * 4

    combined_image = Image.new('RGB', (total_width, total_height))

    for i, img in enumerate(images):
        x_offset = (i % 2) * max(widths)
        y_offset = (i // 2) * max(heights)
        combined_image.paste(img, (x_offset, y_offset))
    
    combined_save_path = "outputs/combined_task1.png"
    combined_image.save(combined_save_path)
    combined_image.show()

combine_task1_images()


In [ ]:
import os
import matplotlib.pyplot as plt
from PIL import Image

def combine_task2_images():
    image_paths = [
        "outputs/accuracy_test.png",
        "outputs/accuracy_val.png",
        "outputs/f1_score_test.png",
        "outputs/f1_score_val.png",
        "outputs/precision_test.png",
        "outputs/precision_val.png",
        "outputs/recall_test.png",
        "outputs/recall_val.png"
    ]
    
    images = [Image.open(image) for image in image_paths]
    widths, heights = zip(*(img.size for img in images))

    total_width = max(widths) * 2
    total_height = max(heights) * 4

    combined_image = Image.new('RGB', (total_width, total_height))

    for i, img in enumerate(images):
        x_offset = (i % 2) * max(widths)
        y_offset = (i // 2) * max(heights)
        combined_image.paste(img, (x_offset, y_offset))
    
    combined_save_path = "outputs/combined_task2.png"
    combined_image.save(combined_save_path)
    combined_image.show()

combine_task2_images()


In [ ]:
from PIL import Image, ImageOps

def combine_tsne_images_tightly():
    image_paths = [
        "outputs/tsne_visualization_batch_1.png",
        "outputs/tsne_visualization_batch_2.png",
        "outputs/tsne_visualization_batch_3.png",
        "outputs/tsne_visualization_batch_4.png",
        "outputs/tsne_visualization_batch_5.png",
        "outputs/tsne_visualization_batch_6.png",
        "outputs/tsne_visualization_batch_7.png",
        "outputs/tsne_visualization_batch_8.png",
        "outputs/tsne_visualization_batch_9.png",
        "outputs/tsne_visualization_batch_10.png"
    ]
    
    images = [ImageOps.crop(Image.open(path), border=90) for path in image_paths]
    
    img_width, img_height = images[0].size
    
    total_width = img_width * 2
    total_height = img_height * 5
    
    combined_image = Image.new('RGB', (total_width, total_height))
    
    for i, img in enumerate(images):
        x_offset = (i // 5) * img_width
        y_offset = (i % 5) * img_height
        combined_image.paste(img, (x_offset, y_offset))
    
    combined_image.save("outputs/tsne_visualization_combined.png")
    combined_image.show()

combine_tsne_images_tightly()
